# 地方競馬 〜 今日のレースを新しいモデルで振り返る（記録のみ・通知なし・購入なし）

自動予想が記録した**その日の4分前のオッズ**と結果を使い、同じレースを

- **新しいモデル**（Input の学習結果の本番のモデル）
- **旧モデル**（同じ学習結果の影のモデル＝これまでの本番）

の両方で予想し直して、◎・単勝の買い目（◎・期待値1.05以上）・ワイド（度合い1.6以上）の成績を並べる。

**Input**
1. 新しい学習用ノートブックの Output（keiba_bundle）
2. その日に動いた**自動予想ノートブックの Output**（auto_records_日付.csv が入っているもの）。Add Input → Your Work → keiba_nar_auto

**Internet**: オン（出馬表を1レース1回だけ取り直す。間隔は自動予想と同じ）。
**注意**: 自動予想が動いている間は実行しない（netkeiba へのアクセスを2本にしない）。その日の自動予想が終わってから実行する。
**実行**: 普通に Run All でよい（Commit でなくてよい）。1レース約4秒 × レース数。

In [ ]:
# ===== ここだけ必要に応じて変更 =====
TARGET_VENUES   = None     # None = 全競馬場。絞るときは ["大井", "川崎", "船橋", "浦和"] のように書く
MINUTES_BEFORE  = 4        # 発走の何分前にオッズを取得して予想するか（この予想で #買い目 に通知＝手動で買う用）
LATE_MINUTES    = 3        # 自動購入を想定した2回目の予想（発走の何分前）。記録と通知のみで #買い目 には送らない。None で無効
LATE_NOTIFY_ALL = True     # True: 2回目の予想を全レース #自動購入想定 に送る（買い目なしも）。False: 買い目があるレースだけ
TAN_TH = 1.05          # 単勝: ◎ かつ 期待値がこの値以上なら買い（2026/11決定）。99 で単勝は買わない
TAN_PMIN = 0.0         # 単勝: 1着確率の下限（◎だけにするので条件なし）
TAN_FAV_ONLY = True    # 単勝は ◎（1着確率1位）だけを候補にする（False で印を問わない）
SHOW_REASONS = False   # 評価の理由（SHAP値）の表示。自動予想では表示しない
TAN_CAP = 999          # 単勝: オッズの上限（なし）
STAKE           = 100      # 1点あたりの金額（記録・集計用）
WIDE_ODDS_FETCH = True     # 4分前に本物のワイドオッズ（人気順の一覧）を1レース1回取得し、実オッズ・実期待値を表示・記録する（False で取得しない）
APT_NOTIFY      = True     # 影のモデル（③上位）の予想と買い目を #新モデル（Secrets: DISCORD_WEBHOOK_APT）に送る。記録は常に残す
WIDE_TH         = 1.6      # ワイドの買い条件: 度合いがこの値以上（3年検証: 1.5以上で105%、1.6以上で108%）
WIDE_MAX        = 2        # ワイドの最大点数（単勝は最大1点。合計で最大3点）
TEST_NOW        = False    # True: 発走時刻を待たずに、これから発走する2レースだけ今すぐ予想・通知して終了（動作確認用）
RESULT_AFTER    = 18       # 発走の何分後に結果を見に行くか
RESULT_RETRY    = 10       # 結果がまだ無いとき、何分おきに見直すか
RESULT_TRIES    = 4        # 何回見ても無ければ「取得できず」として次へ（18分後＋10分×3＝最大48分後まで）
RUN_HOURS       = 11.5     # 起動からこの時間で終了（Kaggleの12時間制限より前に、まとめを送って終わる）

REPLAY_DATE = None   # 振り返る日（例 "20261121"）。None = 今日

In [ ]:
import os, glob, re, io, warnings
import numpy as np, pandas as pd
import lightgbm as lgb, joblib
warnings.filterwarnings("ignore")
DATA_ROOT = "/kaggle/input"

found = sorted(glob.glob(f"{DATA_ROOT}/**/keiba_bundle/meta.joblib", recursive=True))
assert found, "keiba_bundle が見つかりません。学習用ノートブックの Output を Input に追加してください"
BUNDLE = os.path.dirname(found[-1])
meta = joblib.load(f"{BUNDLE}/meta.joblib")
globals().update(meta)
HIST_DB = pd.read_pickle(f"{BUNDLE}/hist.pkl")
HIST_DB["date"] = pd.to_datetime(HIST_DB["date"])
print(f"読み込み完了: モデル作成 {meta['built_at']} / データの最終日 {meta['data_last']} / 過去走 {len(HIST_DB):,} 行")

def combine(d, w):
    s = w[0] * np.log(d["p_win_n"].clip(1e-6)) + w[1] * np.log(d["p_mkt"].clip(1e-6))
    e = np.exp(s - s.groupby(d["race_id"]).transform("max"))
    return e / e.groupby(d["race_id"]).transform("sum")

def norm_name(s):
    return s.astype(str).str.replace(r"[\s\u3000▲△☆◇★]", "", regex=True)

In [ ]:
import os, re, time, json, random, glob, signal, functools, datetime as dt
import requests, pandas as pd
print = functools.partial(print, flush=True)   # ログをため込まず、その場で表示する
from bs4 import BeautifulSoup
from urllib import robotparser

JST = dt.timezone(dt.timedelta(hours=9))
START_DATE = "2025-12-28"            # Kaggleデータの続き
END_DATE   = None                    # None = 昨日まで（JST）
STOP_AT    = (dt.datetime.now(JST) + dt.timedelta(hours=11, minutes=30)).strftime("%Y-%m-%d %H:%M")  # 開始から11.5時間で停止
DELAY, JITTER = 3.5, 1.0             # 1リクエストごとに 3.5〜4.5秒 待つ
OUT = "/kaggle/working"
UA = "Mozilla/5.0 (compatible; personal-research; low-rate single-thread)"
BASE = "https://nar.netkeiba.com"
LIST_URLS  = [BASE + "/top/race_list_sub.html?kaisai_date={d}", BASE + "/top/race_list.html?kaisai_date={d}"]
RESULT_URL = BASE + "/race/result.html?race_id={rid}"

VENUE = {"30": "門別", "35": "盛岡", "36": "水沢", "42": "浦和", "43": "船橋", "44": "大井", "45": "川崎",
         "46": "金沢", "47": "笠松", "48": "名古屋", "50": "園田", "51": "姫路", "53": "福山", "54": "高知",
         "55": "佐賀", "56": "荒尾"}           # 65（帯広ばんえい）は対象外
F_RACES   = f"{OUT}/keiba_races_2026.csv"
F_RESULTS = f"{OUT}/keiba_results_2026.csv"
F_PAYOUTS = f"{OUT}/keiba_payouts_2026.csv"
F_PROG    = f"{OUT}/scrape_progress.json"
COLS_RACES   = ["race_id", "date", "venue", "race_name", "race_number", "course_type", "distance", "turn",
                "weather", "track_condition", "race_class"]
COLS_RESULTS = ["race_id", "rank", "frame", "number", "horse_id", "horse_name", "sex_age", "weight", "jockey_id",
                "jockey_name", "trainer", "time", "margin", "passing", "last_3f", "odds", "popularity", "horse_weight"]
COLS_PAYOUTS = ["race_id", "bet_type", "horse_num", "payout", "popularity"]

class Blocked(Exception):
    pass

class _HardTimeout(Exception):
    pass

def _on_alarm(signum, frame):
    raise _HardTimeout("通信が60秒以上応答なし")

signal.signal(signal.SIGALRM, _on_alarm)

# ---------- 低頻度アクセス ----------
session = requests.Session()
session.headers.update({"User-Agent": UA, "Accept-Language": "ja"})
_last = [0.0]
n_requests = [0]
rp, crawl_delay = None, 0.0

def polite_get(url):
    if rp is not None and not rp.can_fetch(UA, url):
        raise PermissionError(f"robots.txt で禁止されているため取得しません: {url}")
    for attempt in range(2):
        wait = max(DELAY, crawl_delay) + random.uniform(0, JITTER) - (time.time() - _last[0])
        if wait > 0:
            time.sleep(wait)
        signal.alarm(60)   # 1回の通信が60秒を超えたら強制的に打ち切る（固まり防止）
        try:
            r = session.get(url, timeout=(10, 20))
        except (requests.RequestException, _HardTimeout) as e:
            _last[0] = time.time()
            if attempt == 0:
                print("  通信エラー、60秒後に1回だけ再試行:", e)
                time.sleep(60)
                continue
            raise
        finally:
            signal.alarm(0)
        _last[0] = time.time()
        n_requests[0] += 1
        if r.status_code in (403, 429, 503):
            if attempt == 0:
                print(f"  HTTP {r.status_code}（アクセス制限の可能性）。10分休止します")
                time.sleep(600)
                continue
            raise Blocked(f"HTTP {r.status_code} が続いたため、サイトへの負担を避けて停止します")
        return r

def decode(r):
    m = re.search(rb'charset=["\']?([A-Za-z0-9_-]+)', r.content[:5000])
    enc = m.group(1).decode() if m else (r.encoding or "utf-8")
    try:
        return r.content.decode(enc, errors="replace")
    except LookupError:
        return r.content.decode("euc-jp", errors="replace")

def load_robots():
    global rp, crawl_delay
    r = polite_get(BASE + "/robots.txt")
    rp = robotparser.RobotFileParser()
    if r.status_code == 200:
        rp.parse(decode(r).splitlines())
        crawl_delay = float(rp.crawl_delay(UA) or 0)
        print("robots.txt を確認しました。Crawl-delay:", crawl_delay or "指定なし")
    elif r.status_code == 404:
        rp.parse([])
        print("robots.txt はありません（制限なし）")
    else:
        raise Blocked(f"robots.txt が確認できない（HTTP {r.status_code}）ため、安全側に倒して停止します")
    for u in [LIST_URLS[0].format(d="20261015"), RESULT_URL.format(rid="202644101510")]:
        print(("  取得可: " if rp.can_fetch(UA, u) else "  取得不可: ") + u)

# ---------- 解析 ----------
RANK_MAP = {"取消": "取", "中止": "中", "除外": "除", "失格": "失"}
TRAINER_PREFIX = r"^(美浦|栗東|北海道|門別|岩手|盛岡|水沢|浦和|船橋|大井|川崎|金沢|笠松|愛知|名古屋|兵庫|園田|姫路|高知|佐賀|地方|海外|JRA)\s*"
HEAD_KEYS = [("着順", "rank"), ("馬番", "number"), ("馬名", "horse_name"), ("性齢", "sex_age"), ("斤量", "weight"),
             ("騎手", "jockey_name"), ("タイム", "time"), ("着差", "margin"), ("単勝", "odds"), ("オッズ", "odds"),
             ("人気", "popularity"), ("後3F", "last_3f"), ("上り", "last_3f"), ("上がり", "last_3f"),
             ("コーナー", "passing"), ("通過", "passing"), ("厩舎", "trainer"), ("調教師", "trainer"),
             ("馬体重", "horse_weight")]
BET_NORM = {"3連複": "三連複", "3連単": "三連単"}
BET_GROUP = {"単勝": 1, "複勝": 1, "枠連": 2, "馬連": 2, "ワイド": 2, "馬単": 2, "枠単": 2, "三連複": 3, "三連単": 3}
BET_ORDERED = {"単勝", "複勝", "馬単", "枠単", "三連単"}

def map_head(h):
    h = re.sub(r"\s", "", h)
    if h.startswith("枠"):
        return "frame"
    for k, v in HEAD_KEYS:
        if k in h:
            return v
    return None

def clean(s):
    return re.sub(r"\s+", " ", s or "").strip()

TRACK_NORM = {"稍": "稍重", "不": "不良"}

def corner_positions(text):
    # "8,3,(5,6)-4=7" → {"8":1, "3":2, "5":3, "6":3, "4":5, "7":6}（括弧内は同じ位置）
    pos, n = {}, 0
    for grp, single in re.findall(r"\(([^)]*)\)|(\d+)", text):
        nums = re.findall(r"\d+", grp) if grp else [single]
        for x in nums:
            pos[x] = n + 1
        n += len(nums)
    return pos

def parse_list(html, date_str):
    mmdd = date_str[4:]
    rids = sorted(set(re.findall(r"race_id=(\d{12})", html)))
    return [r for r in rids if r[:4] == date_str[:4] and r[6:10] == mmdd and r[4:6] in VENUE]

def parse_result(html, rid):
    soup = BeautifulSoup(html, "html.parser")
    table, heads = None, None
    for t in soup.find_all("table"):
        hs = [clean(th.get_text()) for th in t.find_all("th")]
        if any("馬名" in h for h in hs) and any("着" in h for h in hs):
            table, heads = t, hs
            break
    if table is None:
        return None   # 結果未確定・中止など
    keys = [map_head(h) for h in heads]
    rows = []
    for tr in table.find_all("tr"):
        tds = tr.find_all("td")
        if not tds:
            continue
        if len(tds) != len(keys):
            continue
        rec = {}
        for k, td in zip(keys, tds):
            if k is None or k in rec:
                continue
            if k in ("horse_name", "jockey_name") and td.find("a"):
                rec[k] = clean(td.find("a").get_text())
            else:
                rec[k] = clean(td.get_text(" "))
        rh = str(tr)
        mh = re.search(r"/horse/(\w{10})", rh)
        mj = re.search(r"/jockey/(?:result/recent/|profile/)?(\w{5})(?:/|\b)", rh)
        rec["horse_id"] = mh.group(1) if mh else ""
        rec["jockey_id"] = mj.group(1) if mj else ""
        rows.append(rec)
    if not rows:
        return "CANCELLED"   # 結果表の枠だけで中身が無い = 競走中止・不成立
    res = pd.DataFrame(rows)
    for c in COLS_RESULTS:
        if c not in res:
            res[c] = ""
    res["race_id"] = rid
    res["rank"] = res["rank"].map(lambda x: next((v for k, v in RANK_MAP.items() if str(x).startswith(k)), x))
    res["rank"] = res["rank"].str.replace(r"^(\d+)\s*\(?降\)?$", r"\1(降)", regex=True)
    res["jockey_name"] = res["jockey_name"].str.replace(r"[▲△☆◇★]", "", regex=True).str.strip()
    res["trainer"] = res["trainer"].str.replace(TRAINER_PREFIX, "", regex=True).str.strip()
    res["weight"] = res["weight"].map(lambda x: re.sub(r"\.0$", "", clean(x)))
    res["odds"] = res["odds"].str.replace(",", "", regex=False)
    res["horse_weight"] = res["horse_weight"].str.replace(" ", "", regex=False)
    res["horse_name"] = res["horse_name"].str.replace(r"\s", "", regex=True)

    # 通過順: 結果表に無ければ「コーナー通過順位」表から各馬の位置を組み立てる（例 "4-3-2"）
    corners = []
    for tr in soup.find_all("tr"):
        th, td = tr.find("th"), tr.find("td")
        if th and td and re.search(r"\dコーナー", th.get_text()):
            corners.append(corner_positions(td.get_text(" ")))
    if corners and (res["passing"].fillna("") == "").all():
        res["passing"] = res["number"].map(lambda n: "-".join(str(c[n]) for c in corners if n in c))

    # レース条件
    box = soup.select_one(".RaceData01")
    txt = clean(box.get_text(" ")) if box else clean(soup.get_text(" "))[:5000]
    m = re.search(r"(ダ|芝)\s*(\d{3,4})\s*m", txt)
    mt = re.search(r"\(\s*(右|左|直)", txt)
    mw = re.search(r"天候\s*[:：]\s*([^\s/]+)", txt)
    mb = re.search(r"馬場\s*[:：]\s*([^\s/]+)", txt)
    name = soup.select_one(".RaceName")
    race = {
        "race_id": rid, "date": f"{rid[:4]}-{rid[6:8]}-{rid[8:10]}", "venue": VENUE.get(rid[4:6], ""),
        "race_name": clean(name.get_text()) if name else "", "race_number": int(rid[10:12]),
        "course_type": m.group(1) if m else "", "distance": m.group(2) if m else "",
        "turn": mt.group(1) if mt else "", "weather": mw.group(1) if mw else "",
        "track_condition": TRACK_NORM.get(mb.group(1), mb.group(1)) if mb else "", "race_class": "",
    }

    # 払戻
    pays = []
    for tr in soup.find_all("tr"):
        th = tr.find("th")
        if not th:
            continue
        bt = clean(th.get_text())
        bt = BET_NORM.get(bt, bt)
        if bt not in BET_GROUP:
            continue
        tds = tr.find_all("td")
        if len(tds) < 2:
            continue
        nums = re.findall(r"\d+", tds[0].get_text(" "))
        rest = " ".join(td.get_text(" ") for td in tds[1:])
        yen = [p.replace(",", "") for p in re.findall(r"([\d,]+)\s*円", rest)]
        pop = re.findall(r"(\d+)\s*人気", rest)
        g = BET_GROUP[bt]
        if not yen or len(nums) != len(yen) * g:
            continue
        sep = " → " if bt in BET_ORDERED and g > 1 else " - "
        for i, y in enumerate(yen):
            pays.append({"race_id": rid, "bet_type": bt, "horse_num": sep.join(nums[i * g:(i + 1) * g]),
                         "payout": y, "popularity": pop[i] if i < len(pop) else ""})
    return race, res[COLS_RESULTS], pd.DataFrame(pays, columns=COLS_PAYOUTS)

# ---------- 保存と再開 ----------
def append_csv(df, path, cols):
    if len(df):
        df[cols].to_csv(path, mode="a", index=False, header=not os.path.exists(path), encoding="utf-8")

def load_progress():
    prog = {"dates": {}, "done": [], "skipped": []}
    # 以前の Commit の出力を Input に追加していれば、そこから続きを再開
    for p in glob.glob("/kaggle/input/**/scrape_progress.json", recursive=True) + [F_PROG]:
        if os.path.exists(p):
            prog = json.load(open(p))
            src_dir = os.path.dirname(p)
            if src_dir != OUT:
                for f in ["keiba_races_2026.csv", "keiba_results_2026.csv", "keiba_payouts_2026.csv"]:
                    if os.path.exists(f"{src_dir}/{f}") and not os.path.exists(f"{OUT}/{f}"):
                        pd.read_csv(f"{src_dir}/{f}", dtype=str).to_csv(f"{OUT}/{f}", index=False)
            print(f"進捗を読み込みました: {p}（取得済み {len(prog['done'])} レース）")
    prog.setdefault("skipped", [])
    return prog

def save_progress(prog):
    json.dump(prog, open(F_PROG, "w"), ensure_ascii=False)

print("準備完了")

In [ ]:
from IPython.display import display
pd.set_option("display.unicode.east_asian_width", True)   # 日本語の列幅をそろえる

def show(t):
    # チャット欄の中では表(HTML)が表示されないことがあるため、文字の表で出す
    print(t.to_string(index=False))

BET_TH, BET_CAP = 1.1, 20      # 買い判定: 期待値≧1.1 かつ オッズ≦20倍（検証で決めた固定条件）
BET_PMIN = 0.0                 # 単勝の1着確率の下限（0 = 条件なし。自動予想では①の設定で上書き）
BET_TH, BET_CAP = globals().get("TAN_TH", 1.05), globals().get("TAN_CAP", 999)   # 2026/11 決定: ◎・期待値1.05以上・オッズ上限なし
BET_PMIN = globals().get("TAN_PMIN", 0.0)
BET_FAV = globals().get("TAN_FAV_ONLY", True)   # 単勝は ◎（1着確率1位）だけを候補にする
SHOW_REASONS = globals().get("SHOW_REASONS", True)   # 「評価の理由」（SHAP値）を表示する（自動予想は①で False）
FEAT_JA = {"odds": "単勝オッズ", "venue": "競馬場", "body_weight": "馬体重", "bw_diff": "馬体重の増減", "age": "年齢", "sex": "性別",
           "weight": "斤量", "weight_diff": "斤量の増減", "distance": "距離", "dist_diff": "距離の増減", "days_since_last": "前走からの日数",
           "n_races": "出走数", "frame": "枠番", "number": "馬番", "n_runners": "頭数", "race_number": "レース番号", "month": "月",
           "course_type": "コース", "turn": "回り", "weather": "天気", "track_condition": "馬場",
           "prev_finish_1": "前走の着順", "prev_finish_2": "2走前の着順", "prev_finish_3": "3走前の着順", "prev_finish_ratio": "前走の着順（頭数比）",
           "avg_finish_ratio_5": "近5走の平均着順", "prev_speed": "前走の指数", "avg_speed_5": "近5走の平均指数", "max_speed_5": "近5走の最高指数",
           "top3_rate_5": "近5走の3着内率", "win_rate_10": "近10走の勝率", "prev_time_behind": "前走の着差", "avg_time_behind_5": "近5走の平均着差",
           "prev_last3f_rank": "前走の上がり順位", "avg_last3f_rank_5": "近5走の上がり順位", "prev_corner1_ratio": "前走の1角位置",
           "avg_corner1_ratio_5": "近5走の1角位置（脚質）", "prev_popularity": "前走の人気", "prev_log_odds": "前走のオッズ",
           "same_venue_as_prev": "前走と同じ競馬場", "jockey_key_top3_rate": "騎手の3着内率", "jockey_key_win_rate": "騎手の勝率",
           "trainer_top3_rate": "調教師の3着内率", "max_speed_5_vs_field": "最高指数−メンバー上位", "max_speed_5_gap": "最高指数−メンバー最上位",
           "avg_speed_5_z": "平均指数のメンバー内偏差", "max_speed_5_z": "最高指数のメンバー内偏差", "field_level": "メンバーのレベル",
           "field_vs_prev_field": "前走とのメンバーのレベル差", "avg_field_level_3": "近3走のメンバーのレベル", "apt_course_speed_diff": "コース適性",
           "jt_top3_rate": "騎手×調教師の3着内率", "prev_speed_vs_avg5": "前走の指数−近5走平均", "apt_going_speed_diff": "馬場適性",
           "prev_pace_z": "前走のペース", "prev_l3z": "前走の上がり", "avg_l3z_5": "近5走の上がり", "prev_pace_x_pos": "前走で速いペースを前で受けた",
           "avg_pace_x_pos_5": "近5走で速いペースを前で受けた", "field_front_n": "メンバーの先行馬の数", "front_share": "先行馬の割合",
           "c1_rank_inrace": "メンバー内の位置取り", "front_comp": "先行争いの激しさ"}
def feat_ja(f):
    if f.endswith("_inrace"):
        return FEAT_JA.get(f[:-7], f[:-7]) + "（相対）"
    return FEAT_JA.get(f, f)

def show_reasons(X):
    # モデル（市場と合成する前）の1着確率への寄与（SHAP値）。メンバーの平均との差で「なぜ他の馬より上・下か」を見る
    try:
        cs = np.mean([m.predict(X[features], pred_contrib=True) for m in final_models["win"]], axis=0)[:, :-1]
    except Exception as ex:
        print("（評価の理由は表示できませんでした:", ex, "）"); return
    rel = cs - cs.mean(axis=0, keepdims=True)
    rows = []
    for i in range(len(X)):
        o = np.argsort(rel[i])
        up = [feat_ja(features[j]) for j in o[::-1][:3] if rel[i, j] > 0.02]
        dn = [feat_ja(features[j]) for j in o[:3] if rel[i, j] < -0.02]
        rows.append({"印": X["印"].iloc[i], "馬番": int(X["number"].iloc[i]), "馬名": X["horse_name"].iloc[i],
                     "評価を上げた理由": "・".join(up) or "-", "下げた理由": "・".join(dn) or "-"})
    print("\n▼ 評価の理由（モデルの1着確率への寄与が大きい順。メンバーの平均と比べて）")
    show(pd.DataFrame(rows))
WIDE_RATIO, WIDE_PMIN = 1.6, 0.2   # ワイド: 度合い（モデルのワイド確率÷市場のワイド確率）≧1.6 かつ モデルのワイド確率≧0.2（3年検証で回収率108%）
WIDE_LAM = globals().get("WIDE_LAM", (1.0, 1.0))   # 補正後の確率用の2・3着の補正の強さ（学習ノートブックが保存。無ければ補正なし）
N_SIM = 4000                       # ワイド確率を出すシミュレーション回数
WORK = "/kaggle/working"
HIST_COLS = ["horse_key", "date", "race_id", "number", "finish", "finish_ratio", "speed", "top3", "win",
             "time_behind", "last3f_rank", "corner1_ratio", "popularity", "odds", "distance", "weight", "venue",
             "jockey_key", "trainer", "horse_name"]
B2_HIST = [c for c in ["race_pace_z", "l3z", "pace_x_pos"] if c in HIST_DB.columns]   # B2 ペースと展開（新しい学習結果だけにある）
HIST_COLS = HIST_COLS + B2_HIST
MAIN_KIND = globals().get("main_kind", "cls")          # "soft_init" = 市場を出発点にしたレース単位の学習（2026/11〜）
SHADOW_NAME = globals().get("SHADOW_NAME", "③上位")
print("本番のモデル:", "市場を出発点にしたレース単位の学習（影の6個＋ペースと展開を含む）" if MAIN_KIND == "soft_init" else "1頭ずつ学習（以前の形）",
      f"/ 特徴量 {len(features)}個")
COLMAP = {"枠": "frame", "枠番": "frame", "馬番": "number", "馬名": "horse_name", "性齢": "sex_age", "性別年齢": "sex_age",
          "斤量": "weight", "負担重量": "weight", "騎手": "jockey_name", "騎手名": "jockey_name",
          "調教師": "trainer", "厩舎": "trainer", "馬体重": "horse_weight", "馬体重(増減)": "horse_weight",
          "単勝": "odds", "単勝オッズ": "odds", "オッズ": "odds", "人気": "popularity",
          "着順": "rank", "着": "rank", "タイム": "time", "上がり": "last_3f", "上り": "last_3f", "上がり3F": "last_3f",
          "通過": "passing", "通過順": "passing", "コーナー通過順": "passing"}

# 追加した結果（2026年分など）。前回保存したものが Input にあれば読み込む
extra_hist = HIST_DB[HIST_COLS].iloc[:0].copy()
extra_hist["venue"] = extra_hist["venue"].astype(str)
for p in glob.glob(f"{DATA_ROOT}/**/live_history.csv", recursive=True) + [f"{WORK}/live_history.csv"]:
    if os.path.exists(p):
        extra_hist = pd.read_csv(p, dtype={"horse_key": str, "race_id": str, "jockey_key": str}, parse_dates=["date"])
        print("追加結果を読み込みました:", p, len(extra_hist), "行")
        break
LIVE = {}   # このセッションで予想したレース

def horse_features(d):
    # ④と同じ計算（馬ごとの過去走）
    for col in ["finish", "finish_ratio", "speed", "top3", "win", "time_behind", "last3f_rank", "corner1_ratio",
                "popularity", "odds", "distance", "weight", "number"] + [c for c in B2_HIST if c in d]:
        d[col] = pd.to_numeric(d[col], errors="coerce")
    d["date"] = pd.to_datetime(d["date"])
    d = d.sort_values(["date", "race_id", "number"]).reset_index(drop=True)
    g = d.groupby("horse_key", sort=False)
    def pm(col, n, how="mean"):
        s = g[col].shift(1).groupby(d["horse_key"]).rolling(n, min_periods=1)
        return getattr(s, how)().reset_index(level=0, drop=True)
    for k in (1, 2, 3):
        d[f"prev_finish_{k}"] = g["finish"].shift(k)
    d["prev_finish_ratio"]   = g["finish_ratio"].shift(1)
    d["prev_speed"]          = g["speed"].shift(1)
    d["avg_speed_5"]         = pm("speed", 5)
    d["max_speed_5"]         = pm("speed", 5, "max")
    d["avg_finish_ratio_5"]  = pm("finish_ratio", 5)
    d["top3_rate_5"]         = pm("top3", 5)
    d["win_rate_10"]         = pm("win", 10)
    d["prev_time_behind"]    = g["time_behind"].shift(1)
    d["avg_time_behind_5"]   = pm("time_behind", 5)
    d["prev_last3f_rank"]    = g["last3f_rank"].shift(1)
    d["avg_last3f_rank_5"]   = pm("last3f_rank", 5)
    d["prev_corner1_ratio"]  = g["corner1_ratio"].shift(1)
    d["avg_corner1_ratio_5"] = pm("corner1_ratio", 5)
    d["prev_popularity"]     = g["popularity"].shift(1)
    d["prev_log_odds"]       = np.log1p(g["odds"].shift(1))
    d["n_races"]             = g.cumcount()
    d["days_since_last"]     = (d["date"] - g["date"].shift(1)).dt.days
    d["dist_diff"]           = d["distance"] - g["distance"].shift(1)
    d["weight_diff"]         = d["weight"] - g["weight"].shift(1)
    d["same_venue_as_prev"]  = (d["venue"] == g["venue"].shift(1)).astype(float)
    d["last_date_in_db"]     = g["date"].shift(1)
    if "l3z" in d:                                   # B2 ペースと展開（学習ノートブック ④-3 と同じ計算）
        d["prev_pace_z"]      = g["race_pace_z"].shift(1)
        d["prev_l3z"]         = g["l3z"].shift(1)
        d["avg_l3z_5"]        = pm("l3z", 5)
        d["prev_pace_x_pos"]  = g["pace_x_pos"].shift(1)
        d["avg_pace_x_pos_5"] = pm("pace_x_pos", 5)
    return d

# ---------- 入力の読み取り ----------
def parse_conditions(line):
    c = {}
    m = re.search(r"(\d{4})\s*[-/年.]\s*(\d{1,2})\s*[-/月.]\s*(\d{1,2})", line)
    c["date"] = pd.Timestamp(f"{m[1]}-{int(m[2]):02d}-{int(m[3]):02d}") if m else pd.Timestamp.now(tz="Asia/Tokyo").normalize().tz_localize(None)
    c["venue"] = next((v for v in VENUES if v in line), None)
    m = re.search(r"(\d{1,2})\s*[RrＲ]", line)
    c["race_number"] = int(m[1]) if m else None
    m = re.search(r"(ダート|ダ|芝)\s*(\d{3,4})", line)
    c["course_type"] = ("芝" if m and m[1] == "芝" else "ダ")
    c["distance"] = int(m[2]) if m else None
    m = re.search(r"[右左]", re.sub(r"(ダート|ダ|芝)\s*\d+", "", line))
    c["turn"] = m[0] if m else DEFAULT_TURN.get(c["venue"])
    c["track_condition"] = next((t for t in ["稍重", "不良", "重", "良"] if t in line), None)
    w = line.replace("曇り", "曇")
    c["weather"] = next((x for x in sorted(WEATHERS, key=len, reverse=True) if x in w), "晴")
    missing = [k for k in ["venue", "race_number", "distance", "track_condition"] if c[k] is None]
    if missing:
        raise ValueError(f"1行目から読み取れない項目: {missing}  例) 2026-10-16 大井 11R ダ1800 晴 良")
    c["race_id"] = f"LIVE_{c['date']:%Y%m%d}_{c['venue']}_{c['race_number']:02d}R"
    return c

def parse_table(lines):
    text = "\n".join(l for l in lines if l.strip())
    try:
        t = pd.read_csv(io.StringIO(text), sep=None, engine="python", dtype=str)
        if t.shape[1] < 3:
            raise ValueError
    except Exception:
        t = pd.read_csv(io.StringIO(text), sep=r"\s+", engine="python", dtype=str)
    t.columns = [str(c).strip() for c in t.columns]
    return t.rename(columns=COLMAP)

def norm_s(s):
    return norm_name(s.fillna(""))

def model_proba(m, X):
    # 学習済みモデル（1つ、または乱数違いの複数）の確率の平均（1頭ずつ学習するモデル用: 3着内・影のモデル・以前の本番）
    ms = m if isinstance(m, (list, tuple)) else [m]
    return np.mean([x.predict_proba(X)[:, 1] for x in ms], axis=0)

def win_proba_soft(ms, X):
    # 市場を出発点にしたレース単位のモデル: 市場の確率（締切前のオッズ）に、モデルが学んだ「市場とのずれ」を足す。1レース分の X を渡す
    lp = np.log(np.clip(X["p_mkt"].values.astype(float), 1e-6, None))
    out = []
    for m in ms:
        s_ = np.asarray(m.predict(X[features]), dtype=float) + lp
        e_ = np.exp(s_ - s_.max())
        out.append(e_ / e_.sum())
    return np.mean(out, axis=0)

# ---------- ワイド ----------
def wide_table(X):
    # 1着確率から着順を何千回もシミュレーションし、2頭がそろって複勝圏に入る確率を出す（モデルと市場で同じ乱数）
    n = len(X)
    if n < 5:
        return pd.DataFrame()
    k = 3 if n >= 8 else 2
    gum = np.random.default_rng(0).gumbel(size=(N_SIM, n))
    mats = {}
    for name, col in (("m", "p_comb"), ("k", "p_mkt")):
        g = gum + np.log(np.clip(X[col].values, 1e-9, None))
        top = np.argpartition(-g, k - 1, axis=1)[:, :k]
        hit = np.zeros((N_SIM, n), dtype=np.float32)
        np.put_along_axis(hit, top, 1.0, axis=1)
        mats[name] = (hit.T @ hit) / N_SIM
    # 補正後の確率: 2着・3着を決めるときは強さの差を縮める（人気馬どうしの組を高く見すぎる癖を直した、当たり具合の目安）
    rng_c = np.random.default_rng(1)
    lp = np.log(np.clip(X["p_comb"].values, 1e-9, None))
    rr = np.arange(N_SIM)
    hit = np.zeros((N_SIM, n), dtype=np.float32)
    s = lp[None, :] + rng_c.gumbel(size=(N_SIM, n)); w1 = s.argmax(1); hit[rr, w1] = 1
    s = WIDE_LAM[0] * lp[None, :] + rng_c.gumbel(size=(N_SIM, n)); s[rr, w1] = -np.inf; w2 = s.argmax(1); hit[rr, w2] = 1
    if k == 3:
        s = WIDE_LAM[1] * lp[None, :] + rng_c.gumbel(size=(N_SIM, n)); s[rr, w1] = -np.inf; s[rr, w2] = -np.inf
        hit[rr, s.argmax(1)] = 1
    mats["c"] = (hit.T @ hit) / N_SIM
    iu = np.triu_indices(n, 1)
    nums, names = X["number"].astype(int).values, X["horse_name"].values
    a, b = nums[iu[0]], nums[iu[1]]
    W = pd.DataFrame({"a": np.minimum(a, b), "b": np.maximum(a, b),
                      "馬名": [f"{names[i]}-{names[j]}" for i, j in zip(*iu)],
                      "ワイド確率": mats["m"][iu], "市場のワイド確率": mats["k"][iu], "補正後の確率": mats["c"][iu]})
    W["度合い"] = W["ワイド確率"] / W["市場のワイド確率"]
    W["組み合わせ"] = W["a"].astype(str) + "-" + W["b"].astype(str)
    W["買い"] = ""
    cand = W[(W["度合い"] >= WIDE_RATIO) & (W["ワイド確率"] >= WIDE_PMIN)]
    if len(cand):
        W.loc[cand["度合い"].idxmax(), "買い"] = "ワイド"
    return W

# ---------- 影のモデル（③上位）----------
APT_HIST = all(k in HIST_DB.columns for k in ["course_type", "track_condition", "field_level"]) and "APT_TOP" in globals()
APT_READY = globals().get("final_models_apt") is not None and APT_HIST
print(f"影のモデル（{SHADOW_NAME}）:", "あり（本番と並べて計算します）" if APT_READY else "なし（学習結果に入っていないため本番のモデルだけで予想します）")

def apt_features(X, date):
    # 学習ノートブック ④-2 と同じ計算（そのレースより前の成績だけ）
    cols = ["horse_key", "date", "venue", "course_type", "distance", "track_condition", "speed", "field_level"]
    h = HIST_DB.loc[HIST_DB["horse_key"].isin(X["horse_key"]) & (HIST_DB["date"] < date), cols].sort_values("date").copy()
    h["band"] = pd.cut(h["distance"], APT_BANDS, labels=False).astype("float").fillna(-1)
    h["going"] = h["track_condition"].astype(str).isin(["重", "不良"]).astype(int)
    h["venue"], h["course_type"] = h["venue"].astype(str), h["course_type"].astype(str)
    all_sp = h.groupby("horse_key")["speed"].mean()
    crs = h.groupby(["horse_key", "venue", "course_type", "band"])["speed"].mean()
    gng = h.groupby(["horse_key", "going"])["speed"].mean()
    band = pd.cut(X["distance"], APT_BANDS, labels=False).astype("float").fillna(-1)
    going = X["track_condition"].astype(str).isin(["重", "不良"]).astype(int)
    hk, vn, ct = X["horse_key"], X["venue"].astype(str), X["course_type"].astype(str)
    a_all = hk.map(all_sp)
    a_crs = pd.Series([crs.get((k, v, t, b), np.nan) for k, v, t, b in zip(hk, vn, ct, band)], index=X.index)
    a_gng = pd.Series([gng.get((k, g), np.nan) for k, g in zip(hk, going)], index=X.index)
    X["apt_course_speed_diff"] = a_crs - a_all
    X["apt_going_speed_diff"] = a_gng - a_all
    # 直前の1走の値（その走が欠けていれば欠けたまま）。学習と同じく「最後の行」をそのまま使う
    last_row = h.groupby("horse_key").tail(1).set_index("horse_key")["field_level"]
    avg3 = h.groupby("horse_key").tail(3).groupby("horse_key")["field_level"].mean()
    X["avg_field_level_3"] = hk.map(avg3)
    X["field_vs_prev_field"] = X["field_level"] - hk.map(last_row)
    X["prev_speed_vs_avg5"] = X["prev_speed"] - X["avg_speed_5"]
    return X

# ---------- 予想 ----------
def predict_race(c, e):
    date, rid = c["date"], c["race_id"]
    for k in ["venue", "course_type", "turn", "weather", "track_condition", "distance", "race_number"]:
        e[k] = c[k]
    e["race_id"], e["date"] = rid, date
    for col in ["frame", "sex_age", "weight", "jockey_name", "trainer", "horse_weight"]:
        if col not in e:
            e[col] = np.nan
    # 出走取消・除外の馬（オッズが空欄・0、または「取消」「除外」の記載がある行）は予想の対象から外す
    scratch_word = e.astype(str).apply(lambda r: r.str.contains("取消|除外|取り消し").any(), axis=1)
    for col in ["frame", "number", "weight", "distance", "race_number", "odds"]:
        e[col] = pd.to_numeric(e[col], errors="coerce")
    scratch = scratch_word | e["odds"].isna() | (e["odds"] <= 0) | e["number"].isna()
    if scratch.any():
        print("※ 出走取消・除外として予想から外した馬: " + "、".join(
            f"{'' if pd.isna(n) else int(n)}番 {h}" for n, h in zip(e.loc[scratch, "number"], e.loc[scratch, "horse_name"])))
        e = e[~scratch].copy()
    if len(e) < 2:
        raise ValueError("オッズのある出走馬が2頭未満のため予想できません。単勝オッズの列を確認してください")
    e["sex"] = e["sex_age"].astype(str).str[0]
    e["age"] = pd.to_numeric(e["sex_age"].astype(str).str[1:], errors="coerce")
    bw = e["horse_weight"].astype(str).str.extract(r"(\d+)\s*\(([-+]?\d+)\)")
    e["body_weight"] = pd.to_numeric(bw[0], errors="coerce")
    e["bw_diff"] = pd.to_numeric(bw[1], errors="coerce")
    e["n_runners"], e["month"] = len(e), date.month

    hn, jn, tn = norm_s(e["horse_name"]), norm_s(e["jockey_name"]), norm_s(e["trainer"])
    e["horse_key"] = hn.map(name2horse).fillna("NEW_" + hn)
    e["jockey_key"] = jn.map(name2jockey).fillna("NEWJ_" + jn)
    e["trainer"] = tn.map(name2trainer).fillna(e["trainer"])
    known_h = set(name2horse.index) | set(norm_s(extra_hist["horse_name"]))
    e["DB照合"] = (pd.Series(np.where(hn.isin(known_h), "", "馬×"), index=e.index)
                  + np.where(jn.isin(name2jockey.index) | (jn == ""), "", " 騎手×")
                  + np.where(tn.isin(name2trainer.index) | (tn == ""), "", " 調教師×")).str.strip()

    hist = HIST_DB.loc[HIST_DB["horse_key"].isin(e["horse_key"]) & (HIST_DB["date"] < date), HIST_COLS].copy()
    hist["venue"] = hist["venue"].astype(str)
    xh = extra_hist[extra_hist["horse_key"].isin(e["horse_key"]) & (extra_hist["date"] < date)]
    allr = horse_features(pd.concat([hist, xh, e.drop(columns=["DB照合"])], ignore_index=True))
    X = allr[allr["race_id"] == rid].copy()
    X["DB照合"] = e.set_index("number").loc[X["number"], "DB照合"].values

    past = pd.concat([HIST_DB.loc[HIST_DB["date"] < date, ["jockey_key", "trainer", "win", "top3"]],
                      extra_hist.loc[extra_hist["date"] < date, ["jockey_key", "trainer", "win", "top3"]]])
    past[["win", "top3"]] = past[["win", "top3"]].apply(pd.to_numeric, errors="coerce")
    for key, target in [("jockey_key", "top3"), ("jockey_key", "win"), ("trainer", "top3")]:
        st = past.groupby(key)[target].agg(["sum", "count"])
        pr = PRIOR[target]
        X[f"{key}_{target}_rate"] = (X[key].map(st["sum"]).fillna(0) + pr * K) / (X[key].map(st["count"]).fillna(0) + K)
    for col in ["avg_finish_ratio_5", "top3_rate_5", "avg_time_behind_5", "jockey_key_top3_rate", "avg_speed_5", "max_speed_5"]:
        X[f"{col}_inrace"] = X[col].rank(pct=True)
    # F4 メンバーの中での位置（学習ノートブックと同じ計算）
    for col in ["avg_speed_5", "max_speed_5"]:
        sd_ = X[col].std()
        X[f"{col}_z"] = (X[col] - X[col].mean()) / (sd_ if sd_ and sd_ == sd_ else np.nan)
    X["max_speed_5_gap"] = X["max_speed_5"] - X["max_speed_5"].max()
    X["field_level"] = X["max_speed_5"].nlargest(3).mean()
    X["max_speed_5_vs_field"] = X["max_speed_5"] - X["field_level"]
    if "l3z" in X.columns or "prev_pace_z" in X.columns:   # B2: 今回のメンバーの先行馬の数と自分の位置取り
        front = (X["avg_corner1_ratio_5"] <= 0.3).astype(float)
        X["field_front_n"] = front.sum()
        X["front_share"] = X["field_front_n"] / X["n_runners"]
        X["c1_rank_inrace"] = X["avg_corner1_ratio_5"].rank(pct=True)
        X["front_comp"] = front * (X["field_front_n"] - 1)
    if APT_HIST:
        X = apt_features(X, date)
        p3 = past.assign(_t3=past["top3"].fillna(0)).groupby(["jockey_key", "trainer"])["_t3"].agg(["sum", "count"])
        jt = pd.Series(list(zip(X["jockey_key"], X["trainer"])), index=X.index)
        X["jt_top3_rate"] = ((jt.map(p3["sum"]).fillna(0) + APT_PRI3 * APT_K) / (jt.map(p3["count"]).fillna(0) + APT_K))
    for col in cat_cols:
        bad = set(X[col].dropna().astype(str)) - set(pd.Index(CATS[col]).astype(str))
        if bad:
            print(f"注意: {col} の値 {bad} は学習データに無い表記です")
        X[col] = pd.Categorical(X[col].astype(str), categories=CATS[col])

    missing = [f for f in features if f not in X.columns]
    if missing:
        raise ValueError(f"特徴量が作れていません: {missing}（学習結果と予想ノートブックの版が合っていない可能性があります）")
    X["p_mkt"] = (1 / X["odds"]) / (1 / X["odds"]).sum()
    if MAIN_KIND == "soft_init":
        X["p_win"] = win_proba_soft(final_models["win"], X)
    else:
        X["p_win"] = model_proba(final_models["win"], X[features])
    X["p_win_n"] = X["p_win"] / X["p_win"].sum()
    pt = model_proba(final_models["top3"], X[features])
    X["p_top3"] = np.clip(pt * min(3, len(X)) / pt.sum(), 0, 1)
    X["p_comb"] = combine(X, w_final)
    X["ev"] = X["p_comb"] * X["odds"]
    X = X.sort_values("p_comb", ascending=False).reset_index(drop=True)
    X["印"] = (["◎", "○", "▲", "△", "△"] + [""] * len(X))[:len(X)]
    X["買い"] = ""
    okb = X[(X["ev"] >= BET_TH) & (X["odds"] <= BET_CAP) & (X["p_comb"] >= BET_PMIN)]
    if BET_FAV:
        okb = okb[okb.index == X["p_comb"].idxmax()]   # ◎だけ
    if len(okb):                                   # 条件を満たす馬のうち、期待値が最大の1頭
        X.loc[okb["ev"].idxmax(), "買い"] = "単勝"
    out = pd.DataFrame({
        "印": X["印"], "馬番": X["number"].astype(int), "馬名": X["horse_name"],
        "1着確率": X["p_comb"].round(3), "3着内確率": X["p_top3"].round(3), "単勝": X["odds"],
        "市場の勝率": X["p_mkt"].round(3), "期待値": X["ev"].round(2), "買い": X["買い"],
        "前走(DB)": X["last_date_in_db"].dt.date, "DB照合": X["DB照合"]})
    W = wide_table(X)
    LIVE[rid] = {"cond": c, "entries": e, "pred": out, "wide": W}
    if APT_READY:                                  # 影のモデル: 同じ出馬表・同じオッズで計算
        A = X.copy()
        A["p_win"] = model_proba(final_models_apt["win"], A[features_apt])
        A["p_win_n"] = A["p_win"] / A["p_win"].sum()
        A["p_comb"] = combine(A, w_final_apt)
        A["ev"] = A["p_comb"] * A["odds"]
        A = A.sort_values("p_comb", ascending=False).reset_index(drop=True)
        A["印"] = (["◎", "○", "▲", "△", "△"] + [""] * len(A))[:len(A)]
        A["買い"] = ""
        okb_a = A[(A["ev"] >= BET_TH) & (A["odds"] <= BET_CAP) & (A["p_comb"] >= BET_PMIN)]
        if BET_FAV:
            okb_a = okb_a[okb_a.index == A["p_comb"].idxmax()]
        if len(okb_a):
            A.loc[okb_a["ev"].idxmax(), "買い"] = "単勝"
        LIVE[rid]["pred_apt"] = pd.DataFrame({"印": A["印"], "馬番": A["number"].astype(int), "馬名": A["horse_name"],
                                              "1着確率": A["p_comb"].round(3), "単勝": A["odds"], "期待値": A["ev"].round(2), "買い": A["買い"]})
        LIVE[rid]["wide_apt"] = wide_table(A)
    if len(W):
        W.assign(race=rid, 記録時刻=pd.Timestamp.now(tz="Asia/Tokyo").strftime("%Y-%m-%d %H:%M")).query("買い != ''").to_csv(
            f"{WORK}/wide_log.csv", mode="a", index=False, header=not os.path.exists(f"{WORK}/wide_log.csv"), encoding="utf-8-sig")
    out.assign(race=rid, 記録時刻=pd.Timestamp.now(tz="Asia/Tokyo").strftime("%Y-%m-%d %H:%M")).to_csv(
        f"{WORK}/prediction_log.csv", mode="a", index=False,
        header=not os.path.exists(f"{WORK}/prediction_log.csv"), encoding="utf-8-sig")

    # 返答
    print(f"【予想】{c['date']:%Y-%m-%d} {c['venue']} {c['race_number']}R {c['course_type']}{c['distance']}m "
          f"{c['turn']} {c['weather']} {c['track_condition']}（{len(X)}頭）")
    stale = (date - X["last_date_in_db"]).dt.days
    if (stale > 120).mean() > 0.5:
        print("⚠ 半数以上の馬でDB上の前走が120日以上前です。最近の成績が入っていないので精度は落ちます（結果を貼ると蓄積されます）")
    if (out["DB照合"] != "").any():
        print("⚠ DBで見つからない名前があります（DB照合の列）。表記を確認してください（新馬なら問題なし）")
    buy = out[out["買い"] != ""]
    if len(buy):
        r = buy.iloc[0]
        print(f"→ 買い: {r['馬番']}番 {r['馬名']} 単勝（期待値 {r['期待値']} / {r['単勝']}倍）")
    else:
        print("→ 単勝: 条件（" + ("◎で" if BET_FAV else "") + f"期待値≧{BET_TH}" + (f"・{BET_CAP}倍以下" if BET_CAP < 999 else "") + (f"・1着確率≧{BET_PMIN:.0%}" if BET_PMIN else "") + "）を満たす馬なし：見送り")
    if len(W) and (W["買い"] != "").any():
        r = W[W["買い"] != ""].iloc[0]
        print(f"→ 買い: ワイド {r['組み合わせ']}（{r['馬名']}）度合い {r['度合い']:.2f} / モデル {r['ワイド確率']:.3f} / 市場 {r['市場のワイド確率']:.3f} / 補正後 {r['補正後の確率']:.3f}")
    else:
        print(f"→ ワイド: 条件（度合い≧{WIDE_RATIO}・ワイド確率≧{WIDE_PMIN}）を満たす組み合わせなし：見送り")
    show(out)
    if SHOW_REASONS:
        show_reasons(X)
    if len(W):
        print("\n【ワイド候補】（ワイド確率≧{:.1f} の中で度合いが高い順、上位5点）".format(WIDE_PMIN))
        w5 = W[W["ワイド確率"] >= WIDE_PMIN].sort_values("度合い", ascending=False).head(5)
        if len(w5) == 0:
            w5 = W.sort_values("ワイド確率", ascending=False).head(5)
        show(w5[["組み合わせ", "馬名", "度合い", "ワイド確率", "市場のワイド確率", "補正後の確率", "買い"]].round(3))

# ---------- 結果の取り込み ----------
def add_result(c, t, wide_pays=None):
    global extra_hist
    rid = c["race_id"]
    if rid not in LIVE:
        raise ValueError(f"{rid} はこのセッションで予想していません。先に出馬表を貼って予想してください"
                         "（結果だけ取り込む場合も、出馬表→結果の順で貼る）")
    e = LIVE[rid]["entries"].copy()
    t["number"] = pd.to_numeric(t["number"], errors="coerce")
    e = e.drop(columns=[x for x in ["rank", "time", "last_3f", "passing", "popularity"] if x in e]).merge(
        t[[x for x in ["number", "rank", "time", "last_3f", "passing", "odds", "popularity"] if x in t]].rename(columns={"odds": "odds_res"}),
        on="number", how="left")
    e = e[~e["rank"].astype(str).isin(["取", "除", "nan"])].copy()
    if "odds_res" in e:
        e["odds"] = pd.to_numeric(e["odds_res"], errors="coerce").fillna(e["odds"])
    n = len(e)
    e["n_runners"] = n
    e["finish"] = pd.to_numeric(e["rank"].astype(str).str.extract(r"^(\d+)")[0], errors="coerce").fillna(n).astype(int)
    e["win"], e["top3"] = (e["finish"] == 1).astype(int), (e["finish"] <= 3).astype(int)
    e["finish_ratio"] = (e["finish"] - 1) / max(n - 1, 1)
    e["popularity"] = e["odds"].rank(method="min")
    tm = e.get("time", pd.Series(index=e.index, dtype=str)).astype(str).str.extract(r"(?:(\d+):)?(\d+\.\d)")
    e["time_sec"] = pd.to_numeric(tm[0], errors="coerce").fillna(0) * 60 + pd.to_numeric(tm[1], errors="coerce")
    e["time_behind"] = e["time_sec"] - e["time_sec"].min()
    e["last3f_rank"] = pd.to_numeric(e.get("last_3f"), errors="coerce").rank(pct=True) if "last_3f" in e else np.nan
    if "passing" in e:
        e["corner1_ratio"] = pd.to_numeric(e["passing"].astype(str).str.split("-").str[0], errors="coerce") / n
    else:
        e["corner1_ratio"] = np.nan
    e = e.merge(base, on=BASE_KEYS, how="left")
    e["speed"] = (e["base_mean"] - e["time_sec"]) / e["base_std"] * 10 + 50
    if B2_HIST:                                      # B2 ペースと展開の元になる値（学習ノートブック ④-3 と同じ計算）
        bp = globals().get("BASE_P", {})
        key = (str(c["venue"]), str(c["course_type"]), int(float(c["distance"])), str(c["track_condition"]))
        f_m, f_s, l_m, l_s = bp.get(key, (np.nan,) * 4)
        l3 = pd.to_numeric(e.get("last_3f"), errors="coerce") if "last_3f" in e else pd.Series(np.nan, index=e.index)
        first = e["time_sec"] - l3
        e["race_pace_z"] = (f_m - first[e["finish"] <= 3].median()) / f_s
        e["l3z"] = (l_m - l3) / l_s
        e["pace_x_pos"] = e["race_pace_z"] * (1 - e["corner1_ratio"])

    extra_hist = pd.concat([extra_hist[extra_hist["race_id"] != rid], e[HIST_COLS]], ignore_index=True)
    extra_hist.to_csv(f"{WORK}/live_history.csv", index=False)

    # 答え合わせ
    pred = LIVE[rid]["pred"].merge(e[["number", "finish"]].rename(columns={"number": "馬番", "finish": "着順"}), on="馬番", how="left")
    print(f"【結果を取り込みました】{rid}（DBの追加結果: 累計 {extra_hist['race_id'].nunique()} レース）")
    top = pred.iloc[0]
    print(f"◎ {top['馬番']}番 {top['馬名']} → {int(top['着順']) if pd.notna(top['着順']) else '-'}着")
    buy = pred[pred["買い"] != ""]
    if len(buy):
        r = buy.iloc[0]
        won = r["着順"] == 1
        odds_final = e.loc[e["number"] == r["馬番"], "odds"].iloc[0]
        print(f"買い {r['馬番']}番: {'的中' if won else 'はずれ'}（払戻 {int(odds_final * 100) if won else 0}円 / 100円）")
    show(pred[["印", "馬番", "馬名", "1着確率", "単勝", "買い", "着順"]])
    W = LIVE[rid].get("wide", pd.DataFrame())
    if len(W) and (W["買い"] != "").any():
        r = W[W["買い"] != ""].iloc[0]
        k = 3 if n >= 8 else 2
        fin = e.set_index("number")["finish"]
        hit = all(fin.get(x, 99) <= k for x in (r["a"], r["b"]))
        pay = (wide_pays or {}).get((int(r["a"]), int(r["b"])))
        msg = f"買い ワイド {r['組み合わせ']}: {'的中' if hit else 'はずれ'}"
        if hit:
            msg += f"（払戻 {pay}円 / 100円）" if pay else "（払戻額は「ワイド 3-5 540」の形で結果に添えると記録されます）"
        print(msg)
        LIVE[rid]["wide_result"] = {"hit": hit, "pay": (pay or 0) if hit else 0, "pay_known": (pay is not None) or not hit}

# ---------- 振り返り ----------
def show_log():
    if not os.path.exists(f"{WORK}/live_history.csv") or not LIVE:
        print("まだ結果が取り込まれていません")
        return
    rows = []
    for rid, v in LIVE.items():
        h = extra_hist[extra_hist["race_id"] == rid]
        if h.empty:
            continue
        p = v["pred"].merge(h[["number", "finish", "odds"]].rename(columns={"number": "馬番"}), on="馬番")
        b = p[p["買い"] != ""]
        rows.append({"レース": rid, "◎着順": p.iloc[0]["finish"], "買い": len(b),
                     "払戻": (b["odds"] * 100 * (b["finish"] == 1)).sum()})
    r = pd.DataFrame(rows)
    print(f"結果取り込み済み {len(r)} レース / ◎1着 {(r['◎着順'] == 1).sum()} / ◎3着内 {(r['◎着順'] <= 3).sum()}")
    if r["買い"].sum():
        print(f"買い {int(r['買い'].sum())} 回 / 払戻 {int(r['払戻'].sum())} 円 / 回収率 {r['払戻'].sum() / (100 * r['買い'].sum()):.1%}")
    show(r)
    wr = [v["wide_result"] for v in LIVE.values() if "wide_result" in v]
    if wr:
        nh = sum(x["hit"] for x in wr)
        msg = f"ワイド {len(wr)} 回 / 的中 {nh} 回（{nh / len(wr):.0%}）"
        if all(x["pay_known"] for x in wr):
            tot = sum(x["pay"] for x in wr)
            msg += f" / 払戻 {tot} 円 / 回収率 {tot / (100 * len(wr)):.1%}"
        else:
            msg += " / 払戻額が未入力のレースがあるため回収率は計算できません"
        print(msg)

HELP = """使い方（下の欄に貼って「送信」）

■ 予想: 1行目にレース条件、2行目以降に出馬表（1行目は列名）
2026-10-16 大井 11R ダ1800 晴 良
馬番,馬名,性齢,斤量,騎手,調教師,馬体重,単勝
1,〇〇〇,牡4,56,△△,□□,468(+4),5.2
  - 必須の列は「馬番・馬名・単勝」。他は無くても動く（あるほど精度が上がる）
  - カンマ区切り・タブ区切り（Excel等から貼り付け）・スペース区切りのどれでもOK
  - 日付を省くと今日の日付。回り(右/左)を省くと競馬場の標準
  - オッズが変わったら同じものを貼り直せば再計算

■ 結果: 1行目を「結果」で始めて同じレース条件、2行目以降に着順表
結果 2026-10-16 大井 11R ダ1800 晴 良
着順,馬番,タイム,上がり,通過,単勝
1,5,1:52.3,38.1,2-2-1,4.8
  - 必須は「着順・馬番」。タイム・上がり・通過があると次回以降の予想に使われる
  - ワイドの払戻は、表の下に「ワイド 3-5 540」の形で1行ずつ添えると回収率に反映される（任意）
  - 取り込んだ結果は live_history.csv に保存（次回セッションに引き継ぐ方法は説明参照）

■ 成績: 「成績」と送ると、このセッションの予想の答え合わせ一覧
■ ヘルプ: 「ヘルプ」でこの説明
"""

def respond(text):
    lines = [l for l in text.strip().splitlines() if l.strip()]
    if not lines or lines[0].strip() in ("ヘルプ", "help", "?"):
        print(HELP); return
    if lines[0].strip() in ("成績", "ログ"):
        show_log(); return
    first = lines[0]
    if first.lstrip().startswith("結果"):
        c = parse_conditions(first)
        wide_pays, body = {}, []
        for l in lines[1:]:
            m = re.match(r"\s*ワイド\s*(\d+)\s*[-ー－→]\s*(\d+)\s+([\d,]+)", l)
            if m:
                a, b = sorted((int(m[1]), int(m[2])))
                wide_pays[(a, b)] = int(m[3].replace(",", ""))
            else:
                body.append(l)
        t = parse_table(body)
        if "rank" not in t or "number" not in t:
            raise ValueError(f"結果には「着順」「馬番」の列が必要です。読み取った列: {list(t.columns)}")
        add_result(c, t, wide_pays)
    else:
        c = parse_conditions(first)
        t = parse_table(lines[1:])
        need = [x for x in ["number", "horse_name", "odds"] if x not in t]
        if need:
            raise ValueError(f"出馬表に必要な列がありません: {need}（列名は 馬番・馬名・単勝）。読み取った列: {list(t.columns)}")
        predict_race(c, t)

def chat(text):
    """チャット欄が表示されない環境用: chat(\"\"\"...\"\"\") で同じことができる"""
    try:
        respond(text)
    except Exception as ex:
        print("⚠", ex)
print("準備完了")

In [ ]:
import requests, json, time as _time
from contextlib import redirect_stdout

# ---------- Discord ----------
def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

HOOK_BET, HOOK_LOG, HOOK_AUTO, HOOK_APT = "#買い目", "#記録", "#自動購入想定", "#新モデル"
HOOK_URL = {HOOK_BET: _secret("DISCORD_WEBHOOK_BET"), HOOK_LOG: _secret("DISCORD_WEBHOOK_LOG"),
            HOOK_AUTO: _secret("DISCORD_WEBHOOK_AUTO"), HOOK_APT: _secret("DISCORD_WEBHOOK_APT")}
if not APT_NOTIFY:
    HOOK_URL[HOOK_APT] = None
AUTO_FALLBACK = not HOOK_URL[HOOK_AUTO]
if AUTO_FALLBACK:                      # 3つ目のチャンネルが無ければ #記録 に送る
    HOOK_URL[HOOK_AUTO] = HOOK_URL[HOOK_LOG]
BET_TH, BET_PMIN, BET_CAP = TAN_TH, TAN_PMIN, TAN_CAP   # 単勝の買い条件（④の既定値を①の設定で上書き）
BET_FAV = TAN_FAV_ONLY
SHADOW_NAME = globals().get("SHADOW_NAME", "③上位")
WIDE_RATIO = WIDE_TH                                      # ワイドの買い条件（度合い）
print(f"ワイドの買い条件: 度合い≧{WIDE_RATIO}・ワイド確率≧{WIDE_PMIN} / 補正後の確率の補正: 2着λ={WIDE_LAM[0]:.2f}・3着λ={WIDE_LAM[1]:.2f}")
print("単勝の買い条件: " + ("◎（1着確率1位）で" if BET_FAV else "") + f"期待値≧{BET_TH}" + (f"・1着確率≧{BET_PMIN:.0%}" if BET_PMIN else "") + (f"・{BET_CAP}倍以下" if BET_CAP < 999 else "") + ("" if BET_FAV else "（条件を満たす馬のうち期待値最大の1頭）"))
print("Discord:", " / ".join(f"{k} " + ("OK" if v else "未設定（画面に表示のみ）") for k, v in HOOK_URL.items() if k not in (HOOK_AUTO, HOOK_APT))
      + f" / {HOOK_AUTO} " + ("未設定 → #記録 に送ります（Secrets に DISCORD_WEBHOOK_AUTO を登録し、このノートブックでチェック）" if AUTO_FALLBACK else "OK"))

print(f"{HOOK_APT}（影のモデル {SHADOW_NAME}）: " + ("なし（学習結果に影のモデルが無い）" if not APT_READY else
      "OK" if HOOK_URL[HOOK_APT] else "記録のみ（通知するには Secrets に DISCORD_WEBHOOK_APT を登録し、①で APT_NOTIFY = True）"))

def discord(hook, text, file_path=None):
    print(f"\n[{hook}]\n" + text)
    url = HOOK_URL.get(hook)
    if not url:
        return
    for chunk in [text[i:i + 1900] for i in range(0, len(text), 1900)] or [""]:
        for _ in range(3):
            try:
                if file_path:
                    with open(file_path, "rb") as f:
                        r = requests.post(url, data={"content": chunk}, files={"file": (os.path.basename(file_path), f)}, timeout=20)
                else:
                    r = requests.post(url, json={"content": chunk}, timeout=20)
                if r.status_code == 429:
                    _time.sleep(float(r.json().get("retry_after", 2)) + 0.5)
                    continue
                break
            except Exception as ex:
                print("Discord送信エラー:", ex)
                _time.sleep(3)
        file_path = None

# ---------- 日程 ----------
SHUTUBA_URL = BASE + "/race/shutuba.html?race_id={rid}"

def now():
    return dt.datetime.now(JST)

def get_schedule(date_str):
    r = polite_get(LIST_URLS[0].format(d=date_str))
    soup = BeautifulSoup(decode(r), "html.parser")
    rows = []
    for li in soup.select("li.RaceList_DataItem"):
        a = li.find("a", href=re.compile(r"race_id=\d{12}"))
        if not a:
            continue
        rid = re.search(r"race_id=(\d{12})", a["href"]).group(1)
        if rid[4:6] not in VENUE:
            continue
        rd = li.select_one(".RaceData")
        tm = re.search(r"(\d{1,2}):(\d{2})", rd.get_text(" ") if rd else "")
        if not tm:
            continue
        start = dt.datetime.strptime(date_str, "%Y%m%d").replace(hour=int(tm[1]), minute=int(tm[2]), tzinfo=JST)
        rows.append({"race_id": rid, "venue": VENUE[rid[4:6]], "R": int(rid[10:12]), "start": start})
    sc = pd.DataFrame(rows)
    if TARGET_VENUES and len(sc):
        sc = sc[sc["venue"].isin(TARGET_VENUES)]
    return sc.sort_values("start").reset_index(drop=True) if len(sc) else sc

# ---------- 出馬表（発売中はオッズ・馬体重も入っている） ----------
def parse_shutuba(html):
    soup = BeautifulSoup(html, "html.parser")
    t = soup.find("table", class_="ShutubaTable")
    if t is None:
        return None, None, "出馬表が見つかりません"
    heads = " ".join(th.get_text("", strip=True) for th in t.find_all("th"))
    live = ("オッズ" in heads) and ("予想オッズ" not in heads)
    rows = []
    for tr in t.select("tr.HorseList"):
        def txt(sel):
            x = tr.select_one(sel)
            return re.sub(r"\s+", " ", x.get_text(" ", strip=True)) if x else ""
        num = txt("td[class^=Umaban]")
        name = tr.select_one(".HorseName a")
        jk = tr.select_one("td.Jockey a")
        tr_a = tr.select_one("td.Trainer a")
        w = re.search(r"(\d{3})\s*\(\s*([+-]?\d+)\s*\)", txt("td.Weight"))
        rows.append({"frame": txt("td[class^=Waku]"), "number": num,
                     "horse_name": (name.get("title") or name.get_text(strip=True)) if name else txt(".HorseName"),
                     "sex_age": txt(".Age"), "weight": re.sub(r"[^\d.]", "", txt("td.Txt_C")) or None,
                     "jockey_name": (jk.get("title") or jk.get_text(strip=True)) if jk else "",
                     "trainer": (tr_a.get("title") or tr_a.get_text(strip=True)) if tr_a else "",
                     "horse_weight": f"{w[1]}({w[2]})" if w else "",
                     "odds": txt("td.Popular.Txt_R")})
    e = pd.DataFrame(rows)
    box = soup.select_one(".RaceData01")
    info = re.sub(r"\s+", " ", box.get_text(" ", strip=True)) if box else ""
    m = re.search(r"(ダ|芝)\s*(\d{3,4})\s*m", info)
    mt = re.search(r"\(\s*(右|左|直)", info)
    mw = re.search(r"天候\s*[:：]\s*([^\s/]+)", info)
    mb = re.search(r"馬場\s*[:：]\s*([^\s/]+)", info)
    ms = re.search(r"(\d{1,2}):(\d{2})\s*発走", info)
    cond = {"course_type": m[1] if m else "ダ", "distance": int(m[2]) if m else None,
            "turn": mt[1] if mt else None,
            "weather": (mw[1] if mw else "晴").replace("曇り", "曇"),
            "track_condition": {"稍": "稍重", "不": "不良"}.get(mb[1], mb[1]) if mb else None,
            "start_hm": (int(ms[1]), int(ms[2])) if ms else None, "live": live}
    return e, cond, None

# ---------- 予想（チャット版と同じ計算を使う） ----------
RECORD_ROWS, BET_ROWS, WIDE_ROWS, RACE_INFO = [], [], [], {}
APT_BET_ROWS = []   # 影のモデル（③上位）の買い目（買わない。記録と #新モデル への通知だけ）
APT_WIDE_ROWS = []  # 影のモデル（③上位）のワイド候補（本番の WIDE_ROWS と同じ形）
WIDE_ALL_ROWS = []  # 全部の組のワイド確率・本物のワイドオッズ（下限・上限）・結果（あとで「本物のオッズでの期待値」の買い方を検証する）
WIDE_ODDS_URL = BASE + "/odds/?type=b5&race_id={rid}&housiki=c99"

def fetch_wide_odds(rid):
    # ワイドの人気順の一覧ページ（全組・下限と上限）を1回だけ取得する。取れなければ空で返して、予想は今までどおり続ける
    try:
        h = decode(polite_get(WIDE_ODDS_URL.format(rid=rid)))
    except Exception as ex:
        print(f"  ワイドオッズを取得できませんでした（予想は続けます）: {ex}")
        return {}
    out = {}
    for row in re.findall(r'<tr>\s*<td class="Ninki">\d+</td>(.*?)</tr>', h, re.S):
        nums = re.findall(r'class="UmaBan">(\d+)<', row)
        od = re.findall(r'class="Odds[^"]*">\s*([\d.]+)\s*<', row)
        if len(nums) == 2 and len(od) >= 2:
            lo, hi = float(od[0]), float(od[1])
            if lo > 0:                                    # 0.0 - 0.0 は取消などでオッズなし
                x, y = sorted(map(int, nums))
                out[f"{x}-{y}"] = (lo, hi)
    return out

def wide_candidates(W, bets, row, rid, rows_list):
    # ワイド候補の上位 WIDE_SHOW 組を表にして rows_list に記録する（本番と影のモデルで共通）
    wl = []
    if len(W):
        wc = W[W["ワイド確率"] >= WIDE_PMIN].sort_values("度合い", ascending=False)
        if len(wc) < WIDE_SHOW:
            wc = pd.concat([wc, W.drop(wc.index).sort_values("ワイド確率", ascending=False)])
        wc = wc.head(WIDE_SHOW)
        bought = {b["買い目"] for b in bets if b["券種"] == "ワイド"}
        wl.append("組  馬名           度合い モデル 市場 補正後")
        for _, w in wc.iterrows():
            buy = w["組み合わせ"] in bought
            wl.append(f"{w['組み合わせ']:<5} {short(w['馬名'], 12):　<12} {w['度合い']:>4.2f}  {w['ワイド確率']:.2f}  {w['市場のワイド確率']:.2f}  {w['補正後の確率']:.2f}"
                      + (" 買" if buy else ""))
            rows_list.append({"date": row["start"].strftime("%Y-%m-%d"), "race_id": rid, "venue": row["venue"], "R": row["R"],
                              "組み合わせ": w["組み合わせ"], "馬名": w["馬名"], "ワイド確率": round(w["ワイド確率"], 3),
                              "市場のワイド確率": round(w["市場のワイド確率"], 3), "補正後の確率": round(w["補正後の確率"], 3), "度合い": round(w["度合い"], 2),
                              "条件": "満たす" if (w["度合い"] >= WIDE_RATIO and w["ワイド確率"] >= WIDE_PMIN) else "満たさない",
                              "買い": buy, "的中": None, "払戻_100円あたり": None})
    return wl
WIDE_SHOW = 5   # #記録 に毎回載せるワイド候補の数（買わない組み合わせも記録して答え合わせする）

def short(s, n=6):
    s = str(s)
    return s if len(s) <= n else s[:n]

def make_bets(out, W):
    # 買い目（単勝最大1点＋ワイド最大 WIDE_MAX 点）
    bets = []
    tb = out[out["買い"] != ""]
    if len(tb):
        b = tb.iloc[0]
        bets.append({"券種": "単勝", "買い目": str(int(b["馬番"])), "根拠": f"{b['単勝']}倍・期待値{b['期待値']:.2f}", "枠": "検証済み"})
    if len(W):
        cand = W[(W["度合い"] >= WIDE_RATIO) & (W["ワイド確率"] >= WIDE_PMIN)].sort_values("度合い", ascending=False).head(WIDE_MAX)
        for k, (_, w) in enumerate(cand.iterrows()):
            bets.append({"券種": "ワイド", "買い目": w["組み合わせ"], "根拠": f"度合い{w['度合い']:.2f}・確率{w['ワイド確率']:.2f}・補正後{w['補正後の確率']:.2f}",
                         "枠": "検証済み" if k == 0 else "2点目（未検証）"})
    return bets

TIMING_MAIN = f"{MINUTES_BEFORE}分前（手動）"
TIMING_LATE = f"{LATE_MINUTES}分前（自動想定）" if LATE_MINUTES else None

def predict_one(row, phase="main"):
    rid = row["race_id"]
    mins = MINUTES_BEFORE if phase == "main" else LATE_MINUTES
    r = polite_get(SHUTUBA_URL.format(rid=rid))
    e, cond, err = parse_shutuba(decode(r))
    if err:
        return "error", err
    if not cond["live"]:
        return "not_live", None
    if cond["start_hm"] and not TEST_NOW:
        new_start = row["start"].replace(hour=cond["start_hm"][0], minute=cond["start_hm"][1])
        if new_start - now() > dt.timedelta(minutes=mins + 3):
            return "delayed", new_start
    if phase == "late":
        return predict_late(row, e, cond)
    venue = row["venue"]
    c = {"date": pd.Timestamp(row["start"].date()), "venue": venue, "race_number": row["R"],
         "course_type": cond["course_type"], "distance": cond["distance"] or 1400,
         "turn": cond["turn"] or DEFAULT_TURN.get(venue, "右"),
         "weather": cond["weather"] if cond["weather"] in WEATHERS else "晴",
         "track_condition": cond["track_condition"] or "良"}
    c["race_id"] = f"LIVE_{c['date']:%Y%m%d}_{venue}_{row['R']:02d}R"
    buf = io.StringIO()
    with redirect_stdout(buf):
        try:
            predict_race(c, e.copy())
            ok = True
        except Exception as ex:
            ok, msg = False, str(ex)
    if not ok:
        return "error", msg
    notes = [l for l in buf.getvalue().splitlines() if l.startswith("※")]
    out, W = LIVE[c["race_id"]]["pred"], LIVE[c["race_id"]]["wide"]
    bets = make_bets(out, W)
    RACE_INFO[rid] = {"row": row, "c": c, "out": out, "bets": bets, "pred_time": now().strftime("%H:%M"), "late": None}
    out_a, W_a = LIVE[c["race_id"]].get("pred_apt"), LIVE[c["race_id"]].get("wide_apt")
    apt_map = out_a.set_index("馬番")[["1着確率", "期待値", "買い"]].to_dict("index") if out_a is not None else {}
    for _, o in out.iterrows():
        RECORD_ROWS.append({"date": row["start"].strftime("%Y-%m-%d"), "race_id": rid, "venue": venue, "R": row["R"],
                            "start": row["start"].strftime("%H:%M"), "予想時刻": RACE_INFO[rid]["pred_time"],
                            "印": o["印"], "馬番": int(o["馬番"]), "馬名": o["馬名"], "1着確率": o["1着確率"],
                            "3着内確率": o["3着内確率"], "単勝_予想時": o["単勝"], "期待値": o["期待値"],
                            "単勝買い": o["買い"], "単勝_2回目": None, "1着確率_2回目": None, "期待値_2回目": None,
                            "1着確率_③": apt_map.get(int(o["馬番"]), {}).get("1着確率"),
                            "期待値_③": apt_map.get(int(o["馬番"]), {}).get("期待値"),
                            "単勝買い_③": apt_map.get(int(o["馬番"]), {}).get("買い"),
                            "着順": None, "単勝_確定": None})
    for b in bets:
        BET_ROWS.append({"date": row["start"].strftime("%Y-%m-%d"), "race_id": rid, "venue": venue, "R": row["R"],
                         "タイミング": TIMING_MAIN, "予想時刻": RACE_INFO[rid]["pred_time"],
                         **b, "金額": STAKE, "的中": None, "払戻": None})

    # 本物のワイドオッズ（4分前・1レース1回）。実期待値 = 補正後の確率 × 下限のオッズ（控えめな見積もり）
    if WIDE_ODDS_FETCH and len(W):
        wo = fetch_wide_odds(rid)
        W = W.copy()
        W["実オッズ下限"] = W["組み合わせ"].map(lambda k: wo.get(k, (np.nan, np.nan))[0])
        W["実オッズ上限"] = W["組み合わせ"].map(lambda k: wo.get(k, (np.nan, np.nan))[1])
        W["実期待値"] = W["補正後の確率"] * W["実オッズ下限"]
        W["実期待値_中"] = W["補正後の確率"] * (W["実オッズ下限"] + W["実オッズ上限"]) / 2
        for _, w in W.iterrows():
            WIDE_ALL_ROWS.append({"date": row["start"].strftime("%Y-%m-%d"), "race_id": rid, "venue": venue, "R": row["R"],
                                  "予想時刻": RACE_INFO[rid]["pred_time"], "組み合わせ": w["組み合わせ"],
                                  "ワイド確率": round(w["ワイド確率"], 4), "補正後の確率": round(w["補正後の確率"], 4),
                                  "市場のワイド確率": round(w["市場のワイド確率"], 4), "度合い": round(w["度合い"], 3),
                                  "実オッズ下限": w["実オッズ下限"], "実オッズ上限": w["実オッズ上限"],
                                  "実期待値": None if pd.isna(w["実期待値"]) else round(w["実期待値"], 3),
                                  "的中": None, "払戻_100円あたり": None})
    has_real = "実期待値" in W.columns and W["実期待値"].notna().any()

    # ワイド候補（買い目でなくても毎レース記録する）
    wl = []
    if len(W):
        wc = W[W["ワイド確率"] >= WIDE_PMIN].sort_values("度合い", ascending=False)
        if len(wc) < WIDE_SHOW:
            wc = pd.concat([wc, W.drop(wc.index).sort_values("ワイド確率", ascending=False)]).head(WIDE_SHOW)
        wc = wc.head(WIDE_SHOW)
        bought = {b["買い目"] for b in bets if b["券種"] == "ワイド"}
        wl.append("組  馬名           度合い モデル 市場 補正後" + ("  実オッズ   実期待" if has_real else ""))
        for _, w in wc.iterrows():
            buy = w["組み合わせ"] in bought
            real = (f"  {w['実オッズ下限']:.1f}-{w['実オッズ上限']:.1f} {w['実期待値']:.2f}" if has_real and pd.notna(w.get("実期待値")) else "")
            wl.append(f"{w['組み合わせ']:<5} {short(w['馬名'], 12):　<12} {w['度合い']:>4.2f}  {w['ワイド確率']:.2f}  {w['市場のワイド確率']:.2f}  {w['補正後の確率']:.2f}"
                      + real + (" 買" if buy else ""))
            WIDE_ROWS.append({"date": row["start"].strftime("%Y-%m-%d"), "race_id": rid, "venue": venue, "R": row["R"],
                              "実オッズ下限": w.get("実オッズ下限"), "実オッズ上限": w.get("実オッズ上限"), "実期待値": w.get("実期待値"),
                              "組み合わせ": w["組み合わせ"], "馬名": w["馬名"], "ワイド確率": round(w["ワイド確率"], 3),
                              "市場のワイド確率": round(w["市場のワイド確率"], 3), "補正後の確率": round(w["補正後の確率"], 3), "度合い": round(w["度合い"], 2),
                              "条件": "満たす" if (w["度合い"] >= WIDE_RATIO and w["ワイド確率"] >= WIDE_PMIN) else "満たさない",
                              "買い": buy, "的中": None, "払戻_100円あたり": None})
    if has_real:                                   # 参考: 本物のオッズで見た期待値の高い組（確率0.2以上・まだ買い条件ではない）
        top_real = W[(W["補正後の確率"] >= 0.2) & W["実期待値"].notna()].sort_values("実期待値", ascending=False).head(2)
        if len(top_real):
            wl.append("参考 実期待値の高い組: " + " / ".join(f"{r_['組み合わせ']}（補正後{r_['補正後の確率']:.2f}×{r_['実オッズ下限']:.1f}倍＝{r_['実期待値']:.2f}）" for _, r_ in top_real.iterrows()))
    wtxt = (f"\nワイド候補（買い条件: 度合い≧{WIDE_RATIO}・確率≧{WIDE_PMIN}）\n```\n" + "\n".join(wl) + "\n```") if wl else "\nワイド候補なし（頭数不足）"

    # 通知
    head = (f"🏇 {venue}{row['R']}R {row['start']:%H:%M}発走 {c['course_type']}{c['distance']} "
            f"{c['weather']} {c['track_condition']}（{len(out)}頭）")
    tbl = ["印 番 馬名     勝率  3内  単勝  期待"]
    for _, o in out.iterrows():
        tbl.append(f"{o['印'] or '　'} {int(o['馬番']):>2} {short(o['馬名']):　<6} {o['1着確率']:.3f} {o['3着内確率']:.2f} "
                   f"{o['単勝']:>5.1f} {o['期待値']:>4.2f}" + (" 単" if o["買い"] else ""))
    blines = [f"{b['券種']} {b['買い目']}（{b['根拠']}）" + ("" if b["枠"] == "検証済み" else " ※2点目・未検証枠") for b in bets]
    note = ("\n" + "\n".join(notes)) if notes else ""
    discord(HOOK_LOG, f"{head}\n予想（オッズ {RACE_INFO[rid]['pred_time']}時点）{note}\n```\n" + "\n".join(tbl) + "\n```" + wtxt
            + ("\n買い目: " + " / ".join(blines) if bets else "\n買い目なし（見送り）"))
    if bets:
        discord(HOOK_BET, head + note + "\n" + "\n".join(blines) + f"\n計{len(bets)}点 {len(bets) * STAKE:,}円")
    if out_a is not None:
        notify_apt(rid, row, c, head, out, out_a, W_a if W_a is not None else pd.DataFrame())
    return "done", None

def notify_apt(rid, row, c, head, out, out_a, W_a):
    # 影のモデル（旧モデル）: 予想と買い目を記録し、#新モデル に送る（実際には買わない）
    bets_a = make_bets(out_a, W_a)
    RACE_INFO[rid]["bets_apt"] = bets_a
    for b in bets_a:
        APT_BET_ROWS.append({"date": row["start"].strftime("%Y-%m-%d"), "race_id": rid, "venue": row["venue"], "R": row["R"],
                             "予想時刻": RACE_INFO[rid]["pred_time"], **b, "金額": STAKE, "的中": None, "払戻": None})
    main_p = out.set_index("馬番")["1着確率"]
    tbl = ["印 番 馬名     影勝率 本番  単勝  影期待"]
    for _, o in out_a.iterrows():
        tbl.append(f"{o['印'] or '　'} {int(o['馬番']):>2} {short(o['馬名']):　<6} {o['1着確率']:.3f} {main_p.get(o['馬番'], np.nan):.3f} "
                   f"{o['単勝']:>5.1f} {o['期待値']:>4.2f}" + (" 単" if o["買い"] else ""))
    main_b = {(b["券種"], b["買い目"]) for b in RACE_INFO[rid]["bets"]}
    bl = [f"{b['券種']} {b['買い目']}（{b['根拠']}）" + ("" if (b["券種"], b["買い目"]) in main_b else " ★本番と違う")
          + ("" if b["枠"] == "検証済み" else " ※2点目") for b in bets_a]
    wl = wide_candidates(W_a, bets_a, row, rid, APT_WIDE_ROWS)
    wtxt = (f"\n影のワイド候補（買い条件: 度合い≧{WIDE_RATIO}・確率≧{WIDE_PMIN}）\n```\n" + "\n".join(wl) + "\n```") if wl else "\n影のワイド候補なし（頭数不足）"
    discord(HOOK_APT, f"🧪 {head}\n{SHADOW_NAME}の予想（オッズ {RACE_INFO[rid]['pred_time']}時点・本番の勝率と並べて）\n```\n" + "\n".join(tbl) + "\n```"
            + wtxt + ("\n影の買い目: " + " / ".join(bl) if bets_a else "\n影の買い目なし"))

# ---------- 2回目の予想（自動購入想定）----------
def predict_late(row, e, cond):
    rid = row["race_id"]
    info = RACE_INFO.get(rid)
    if info is None:
        return "error", "1回目の予想がありません"
    c = dict(info["c"])
    c["race_id"] = info["c"]["race_id"] + "_L"          # 1回目の予想を上書きしないよう別名で計算
    buf = io.StringIO()
    with redirect_stdout(buf):
        try:
            predict_race(c, e.copy())
        except Exception as ex:
            return "error", str(ex)
    out2, W2 = LIVE[c["race_id"]]["pred"], LIVE[c["race_id"]]["wide"]
    bets2 = make_bets(out2, W2)
    t2 = now().strftime("%H:%M")
    info["late"] = {"out": out2, "bets": bets2, "time": t2}
    o2i = out2.set_index("馬番")
    o2 = o2i["単勝"]
    for rec in RECORD_ROWS:
        if rec["race_id"] == rid:
            rec["単勝_2回目"] = o2.get(rec["馬番"])
            rec["1着確率_2回目"] = o2i["1着確率"].get(rec["馬番"])
            rec["期待値_2回目"] = o2i["期待値"].get(rec["馬番"])
    for b in bets2:
        BET_ROWS.append({"date": row["start"].strftime("%Y-%m-%d"), "race_id": rid, "venue": row["venue"], "R": row["R"],
                         "タイミング": TIMING_LATE, "予想時刻": t2, **b, "金額": STAKE, "的中": None, "払戻": None})
    # 通知: 2回目の買い目と、1回目との違い・印の馬のオッズの動き
    k1 = {(b["券種"], b["買い目"]) for b in info["bets"]}
    k2 = {(b["券種"], b["買い目"]) for b in bets2}
    blines = [f"{b['券種']} {b['買い目']}（{b['根拠']}）" + ("" if b["枠"] == "検証済み" else " ※2点目") +
              ("" if (b["券種"], b["買い目"]) in k1 else " 🆕") for b in bets2]
    dropped = [f"{k}{m}" for k, m in sorted(k1 - k2)]
    o1 = info["out"].set_index("馬番")["単勝"]
    marks = info["out"][info["out"]["印"] != ""]
    moves = " / ".join(f"{m['印']}{int(m['馬番'])} {o1.get(m['馬番']):.1f}→{o2.get(m['馬番'], float('nan')):.1f}" for _, m in marks.iterrows())
    msg = (f"🤖 {row['venue']}{row['R']}R {row['start']:%H:%M}発走 自動購入想定（オッズ {t2}時点＝{LATE_MINUTES}分前）\n"
           + ("買い目: " + " / ".join(blines) + f"\n計{len(bets2)}点 {len(bets2) * STAKE:,}円" if bets2 else "買い目なし（見送り）")
           + (f"\n{MINUTES_BEFORE}分前から消えた買い目: " + "、".join(dropped) if dropped else "")
           + (f"\nオッズの動き（{info['pred_time']}→{t2}）: " + moves if moves else ""))
    if bets2 or dropped or LATE_NOTIFY_ALL:
        discord(HOOK_AUTO, msg)
    else:
        print(msg)
    return "done", None

# ---------- 結果 ----------
def check_result(rid):
    r = polite_get(RESULT_URL.format(rid=rid))
    parsed = parse_result(decode(r), rid) if r.status_code == 200 else None
    if parsed in (None, "CANCELLED"):
        return parsed
    race, res, pay = parsed
    info = RACE_INFO[rid]
    res = res.copy()
    res["num"] = pd.to_numeric(res["number"], errors="coerce")
    res["fin"] = pd.to_numeric(res["rank"].astype(str).str.extract(r"^(\d+)")[0], errors="coerce")
    fin = res.set_index("num")["fin"]
    odds_f = pd.to_numeric(res.set_index("num")["odds"], errors="coerce")
    for rec in RECORD_ROWS:
        if rec["race_id"] == rid:
            rec["着順"] = fin.get(rec["馬番"])
            rec["単勝_確定"] = odds_f.get(rec["馬番"])
    pay = pay.copy()
    pay["key"] = pay["horse_num"].astype(str).str.findall(r"\d+").map(lambda x: "-".join(sorted(x, key=int)))
    payd = {(b, k): int(p) for b, k, p in zip(pay["bet_type"], pay["key"], pd.to_numeric(pay["payout"], errors="coerce").fillna(0))}
    blines = []
    for b in BET_ROWS:
        if b["race_id"] != rid:
            continue
        key = "-".join(sorted(re.findall(r"\d+", b["買い目"]), key=int))
        p = payd.get((b["券種"], key), 0) * STAKE // 100
        b["的中"], b["払戻"] = p > 0, p
        blines.append((b.get("タイミング", TIMING_MAIN), f"{b['券種']}{b['買い目']} {'的中 ' + format(p, ',') + '円' if p else 'はずれ'}", p))
    ares = []
    for b in APT_BET_ROWS:
        if b["race_id"] != rid:
            continue
        key = "-".join(sorted(re.findall(r"\d+", b["買い目"]), key=int))
        p = payd.get((b["券種"], key), 0) * STAKE // 100
        b["的中"], b["払戻"] = p > 0, p
        ares.append(f"{b['券種']}{b['買い目']} {'的中 ' + format(p, ',') + '円' if p else 'はずれ'}")
    awres = []
    for w in APT_WIDE_ROWS:
        if w["race_id"] != rid:
            continue
        p = payd.get(("ワイド", w["組み合わせ"]), 0)
        w["的中"], w["払戻_100円あたり"] = p > 0, p
        awres.append(f"{w['組み合わせ']}{'(買)' if w['買い'] else ''} {'○ ' + format(p, ',') + '円' if p else '×'}")
    if HOOK_URL.get(HOOK_APT) and info.get("bets_apt") is not None:
        top_n = fin.idxmin() if fin.notna().any() else None
        discord(HOOK_APT, f"🏁 {info['row']['venue']}{info['row']['R']}R 結果: 1着 {'' if top_n is None else int(top_n)}番"
                + (" / 影の買い目: " + " / ".join(ares) if ares else " / 影の買い目なし")
                + ("\n影のワイド候補: " + " / ".join(awres) if awres else ""))
    for w in WIDE_ALL_ROWS:
        if w["race_id"] == rid:
            p = payd.get(("ワイド", w["組み合わせ"]), 0)
            w["的中"], w["払戻_100円あたり"] = p > 0, p
    wres = []
    for w in WIDE_ROWS:
        if w["race_id"] != rid:
            continue
        p = payd.get(("ワイド", w["組み合わせ"]), 0)
        w["的中"], w["払戻_100円あたり"] = p > 0, p
        wres.append(f"{w['組み合わせ']}{'(買)' if w['買い'] else ''} {'○ ' + format(p, ',') + '円' if p else '×'}")
    out = info["out"].copy()
    out["着"] = out["馬番"].map(fin)
    out["確定"] = out["馬番"].map(odds_f)
    late = info.get("late")
    out["2回目"] = out["馬番"].map(late["out"].set_index("馬番")["単勝"]) if late else np.nan
    show_rows = out[(out["着"] <= 5) | (out["印"] != "")].sort_values("着", na_position="last")
    tbl = ["着 番 馬名     印 勝率  " + (f"{MINUTES_BEFORE}分前→{LATE_MINUTES}分前→確定" if late else "予想時→確定")]
    for _, o in show_rows.iterrows():
        f = "" if pd.isna(o["着"]) else int(o["着"])
        tbl.append(f"{f:>2} {int(o['馬番']):>2} {short(o['馬名']):　<6} {o['印'] or '　'} {o['1着確率']:.3f} "
                   f"{o['単勝']:>5.1f}→" + ("" if not late else ("  -  " if pd.isna(o['2回目']) else format(o['2回目'], '>5.1f')) + "→")
                   + ("" if pd.isna(o['確定']) else format(o['確定'], '>5.1f')))
    top = out.iloc[0]
    msg = (f"🏁 {info['row']['venue']}{info['row']['R']}R 結果\n```\n" + "\n".join(tbl) + "\n```\n"
           f"◎{int(top['馬番'])}番 → {'-' if pd.isna(top['着']) else int(top['着'])}着")
    if wres:
        msg += "\nワイド候補: " + " / ".join(wres)
    for tim in dict.fromkeys(t for t, _, _ in blines):
        g = [(x, p) for t, x, p in blines if t == tim]
        msg += f"\n{tim}: " + " / ".join(x for x, _ in g) + f"（{STAKE * len(g):,}円 → {sum(p for _, p in g):,}円）"
    discord(HOOK_LOG, msg)
    return "done"

# ---------- 1日のまとめ ----------
def daily_summary(date_str):
    R = pd.DataFrame(RECORD_ROWS); B = pd.DataFrame(BET_ROWS)
    lines = [f"📊 {date_str[:4]}/{date_str[4:6]}/{date_str[6:]} 成績"]
    if len(R):
        top = R[R["印"] == "◎"]
        done = top["着順"].notna()
        lines.append(f"予想 {R['race_id'].nunique()}レース（結果確認 {int(done.sum())}） / ◎1着 {int((top['着順'] == 1).sum())} / ◎3着内 {int((top['着順'] <= 3).sum())}")
    tot_in = tot_out = 0
    if len(B):
        B2 = B[B["的中"].notna()]
        for tim, gt in B2.groupby("タイミング", sort=False):
            lines.append(f"【{tim}】")
            for (kind, tier), g in gt.groupby(["券種", "枠"]):
                label = kind + ("" if tier == "検証済み" else "（2点目）")
                lines.append(f"  {label} {len(g)}点 的中{int(g['的中'].sum())} 回収率 {g['払戻'].sum() / (STAKE * len(g)):.0%}")
            ti, to = STAKE * len(gt), int(gt["払戻"].sum())
            lines.append(f"  計 {ti:,}円 → {to:,}円（{to - ti:+,}円）")
    else:
        lines.append("買い目なし")
    WR = pd.DataFrame(WIDE_ROWS)
    if len(WR) and WR["的中"].notna().any():
        w2 = WR[WR["的中"].notna()]
        for lab, g in [("ワイド候補 全部", w2), ("うち条件を満たす", w2[w2["条件"] == "満たす"])]:
            if len(g):
                lines.append(f"{lab} {len(g)}組 的中{int(g['的中'].sum())} 仮想回収率 {g['払戻_100円あたり'].sum() / (100 * len(g)):.0%}")
    wide_path = f"{OUT}/auto_wide_{date_str}.csv"
    WR.to_csv(wide_path, index=False, encoding="utf-8-sig")
    rec_path, bet_path = f"{OUT}/auto_records_{date_str}.csv", f"{OUT}/auto_bets_{date_str}.csv"
    late_path = f"{OUT}/auto_bets_late_{date_str}.csv"
    R.to_csv(rec_path, index=False, encoding="utf-8-sig")
    B_main = B[B["タイミング"] == TIMING_MAIN] if len(B) else B
    B_late = B[B["タイミング"] == TIMING_LATE] if len(B) else B
    B_main.to_csv(bet_path, index=False, encoding="utf-8-sig")          # 手動版（今までどおり）
    if TIMING_LATE:
        B_late.to_csv(late_path, index=False, encoding="utf-8-sig")     # 自動購入想定版
    # 累計（以前の Output を Input に追加している場合）
    old = [p for p in glob.glob(f"{DATA_ROOT}/**/auto_bets_*.csv", recursive=True) if date_str not in p]
    if old:
        allb = pd.concat([pd.read_csv(p) for p in old] + [B], ignore_index=True)
        if "タイミング" not in allb:
            allb["タイミング"] = np.nan
        allb["タイミング"] = allb["タイミング"].fillna("5分前（手動）")       # 以前の記録（タイミング列なし）は5分前の手動
        allb = allb.drop_duplicates(["date", "race_id", "タイミング", "券種", "買い目"])
        allb = allb[allb["的中"].notna()]
        for tim, g in allb.groupby("タイミング", sort=False):
            lines.append(f"累計 {tim}（{g['date'].nunique()}日・{len(g)}点）回収率 {g['払戻'].sum() / (STAKE * len(g)):.0%}")
    if WIDE_ALL_ROWS:
        WA = pd.DataFrame(WIDE_ALL_ROWS)
        wa_path = f"{OUT}/auto_wide_all_{date_str}.csv"
        WA.to_csv(wa_path, index=False, encoding="utf-8-sig")
        g = WA[WA["的中"].notna() & WA["実期待値"].notna() & (WA["補正後の確率"] >= 0.2)]
        for th in [1.0, 1.2]:
            s = g[g["実期待値"] >= th]
            if len(s):
                lines.append(f"参考 実期待値{th}以上の組（補正後0.2以上・全部買った場合）: {len(s)}組 的中{int(s['的中'].sum())} 回収率 {s['払戻_100円あたり'].sum() / (100 * len(s)):.0%}")
    discord(HOOK_LOG, "\n".join(lines), rec_path)
    if WIDE_ALL_ROWS:
        discord(HOOK_LOG, "全部の組のワイド確率と本物のワイドオッズの記録", wa_path)
    if APT_READY:
        AB = pd.DataFrame(APT_BET_ROWS)
        apt_path = f"{OUT}/auto_bets_apt_{date_str}.csv"
        AB.to_csv(apt_path, index=False, encoding="utf-8-sig")
        al = [f"📊 {date_str[:4]}/{date_str[4:6]}/{date_str[6:]} 影のモデル（{SHADOW_NAME}）の成績（{TIMING_MAIN}のオッズで判断・実際には買っていない）"]
        G = AB[AB["的中"].notna()] if len(AB) else AB
        if len(G):
            for kind, g in G[G["枠"] == "検証済み"].groupby("券種"):
                al.append(f"{kind} {len(g)}点 的中{int(g['的中'].sum())} 回収率 {g['払戻'].sum() / (STAKE * len(g)):.0%}")
        else:
            al.append("買い目なし")
        if len(R) and "1着確率_③" in R and R["着順"].notna().any():
            rr = R[R["着順"].notna()].copy()
            rr["win"] = (pd.to_numeric(rr["着順"], errors="coerce") == 1).astype(int)
            ok = rr.groupby("race_id")["win"].transform("sum") == 1
            rr = rr[ok & rr["1着確率_③"].notna()]
            if len(rr):
                ll_m = -np.log(rr.loc[rr["win"] == 1, "1着確率"].clip(1e-6)).mean()
                ll_a = -np.log(rr.loc[rr["win"] == 1, "1着確率_③"].astype(float).clip(1e-6)).mean()
                al.append(f"勝ち馬につけた確率の対数損失（小さいほど良い）: 本番 {ll_m:.4f} / 影 {ll_a:.4f}（{rr['race_id'].nunique()}レース）")
        AW = pd.DataFrame(APT_WIDE_ROWS)
        if len(AW):
            aw2 = AW[AW["的中"].notna()]
            for lab, g in [("影のワイド候補 全部", aw2), ("うち条件を満たす", aw2[aw2["条件"] == "満たす"])]:
                if len(g):
                    al.append(f"{lab} {len(g)}組 的中{int(g['的中'].sum())} 仮想回収率 {g['払戻_100円あたり'].sum() / (100 * len(g)):.0%}")
        discord(HOOK_APT, "\n".join(al), apt_path if len(AB) else None)
        if len(AW):
            aw_path = f"{OUT}/auto_wide_apt_{date_str}.csv"
            AW.to_csv(aw_path, index=False, encoding="utf-8-sig")
            discord(HOOK_APT, "影のワイド候補の記録", aw_path)
    if len(B_main):
        discord(HOOK_LOG, f"買い目の記録（{TIMING_MAIN}）", bet_path)
    if TIMING_LATE:
        L = B_late[B_late["的中"].notna()] if len(B_late) else B_late
        al = [f"📊 {date_str[:4]}/{date_str[4:6]}/{date_str[6:]} 自動購入想定（{LATE_MINUTES}分前）の成績"]
        if len(L):
            for (kind, tier), g in L.groupby(["券種", "枠"]):
                al.append(f"{kind}{'' if tier == '検証済み' else '（2点目）'} {len(g)}点 的中{int(g['的中'].sum())} 回収率 {g['払戻'].sum() / (STAKE * len(g)):.0%}")
            al.append(f"計 {STAKE * len(L):,}円 → {int(L['払戻'].sum()):,}円（{int(L['払戻'].sum()) - STAKE * len(L):+,}円）")
        else:
            al.append(f"買い目 {len(B_late)}点（結果未確認）" if len(B_late) else "買い目なし")
        discord(HOOK_AUTO, "\n".join(al), late_path if len(B_late) else None)
    if len(WR):
        discord(HOOK_LOG, "ワイド候補の記録", wide_path)

print("準備完了")

## 振り返り（新旧のモデルで予想し直す）

In [ ]:
import io, glob
from contextlib import redirect_stdout
DATE = REPLAY_DATE or now().strftime("%Y%m%d")
rec_files = sorted(glob.glob(f"{DATA_ROOT}/**/auto_records_{DATE}.csv", recursive=True))
assert rec_files, f"auto_records_{DATE}.csv が見つかりません。その日の自動予想ノートブックの Output を Input に追加してください"
REC = pd.read_csv(rec_files[-1], dtype={"race_id": str})
REC = REC[REC["着順"].notna()].copy()
WPAY = {}
wa = sorted(glob.glob(f"{DATA_ROOT}/**/auto_wide_all_{DATE}.csv", recursive=True))
if wa:
    WA = pd.read_csv(wa[-1], dtype={"race_id": str})
    WPAY = {(r["race_id"], str(r["組み合わせ"])): float(r["払戻_100円あたり"]) for _, r in WA.iterrows() if pd.notna(r["払戻_100円あたり"])}
ab = sorted(glob.glob(f"{DATA_ROOT}/**/auto_bets_{DATE}.csv", recursive=True))
ACT = pd.read_csv(ab[-1], dtype={"race_id": str}) if ab else pd.DataFrame()
last = pd.Timestamp(meta["data_last"])
if last >= pd.Timestamp(DATE):
    print(f"⚠ 学習データの最終日（{last.date()}）が振り返る日以降です。この日の結果を学習に使っているので、新しいモデルが有利に出ます（比較になりません）")
print(f"振り返る日: {DATE} / 結果のあるレース {REC['race_id'].nunique()} / 学習データの最終日 {last.date()} / ワイドの払戻 {'あり' if WPAY else 'なし（的中だけ表示）'}")
print(f"新しいモデル: {'市場を出発点にしたレース単位の学習' if globals().get('main_kind') == 'soft_init' else '1頭ずつ学習'}（特徴量 {len(features)}個）/ 旧モデル: {'あり' if APT_READY else 'なし'}")
assert APT_READY, "学習結果に影のモデル（旧モデル）が入っていません"

load_robots()
RACES = []
for rid, Rr in REC.groupby("race_id", sort=False):
    r = polite_get(SHUTUBA_URL.format(rid=rid))
    e, cond, err = parse_shutuba(decode(r))
    if err:
        print(f"{rid}: {err}"); continue
    e["number"] = pd.to_numeric(e["number"], errors="coerce")
    om = Rr.set_index("馬番")["単勝_予想時"]
    e = e[e["number"].isin(om.index) & e["number"].map(om).notna()].copy()
    e["odds"] = e["number"].map(om).astype(float).map(lambda x: f"{x:.1f}")
    e["number"] = e["number"].astype(int).astype(str)
    row0 = Rr.iloc[0]
    venue, Rn = row0["venue"], int(row0["R"])
    c = {"date": pd.Timestamp(DATE), "venue": venue, "race_number": Rn,
         "course_type": cond["course_type"], "distance": cond["distance"] or 1400,
         "turn": cond["turn"] or DEFAULT_TURN.get(venue, "右"),
         "weather": cond["weather"] if cond["weather"] in WEATHERS else "晴",
         "track_condition": cond["track_condition"] or "良"}
    c["race_id"] = f"REPLAY_{DATE}_{venue}_{Rn:02d}R"
    try:
        with redirect_stdout(io.StringIO()):
            predict_race(c, e.reset_index(drop=True))
    except Exception as ex:
        print(f"{rid}: 予想できず {ex}"); continue
    L = LIVE[c["race_id"]]
    if L.get("pred_apt") is None:
        continue
    fin = Rr.set_index("馬番")["着順"].astype(float)
    fodds = Rr.set_index("馬番")["単勝_確定"].astype(float)
    n_run = len(fin)
    def settle(out, W):
        bets = make_bets(out, W)            # 自動予想と同じ選び方（単勝は最大1点・ワイドは最大 WIDE_MAX 点）
        res = []
        for b in bets:
            if b["券種"] == "単勝":
                n = int(b["買い目"]); hit = fin.get(n) == 1
                res.append({**b, "的中": hit, "払戻": float(fodds.get(n, 0) * 100) if hit else 0.0})
            else:
                a_, b_ = map(int, str(b["買い目"]).split("-"))
                hit = fin.get(a_, 99) <= 3 and fin.get(b_, 99) <= 3
                res.append({**b, "的中": hit, "払戻": WPAY.get((rid, str(b["買い目"])), np.nan if hit else 0.0) if hit else 0.0})
        return res
    RACES.append({"rid": rid, "venue": venue, "R": Rn, "start": row0.get("start", ""), "fin": fin, "fodds": fodds, "n": n_run,
                  "new": L["pred"], "old": L["pred_apt"], "Wn": L["wide"], "Wo": L.get("wide_apt"),
                  "bets_new": settle(L["pred"], L["wide"]), "bets_old": settle(L["pred_apt"], L.get("wide_apt")), "rec": Rr})
print(f"振り返れたレース: {len(RACES)}")

## レースごと（自動予想の #買い目 と同じ中身を、新旧並べて）

In [ ]:
def wide_lines(W, bets, rid, fin):
    if W is None or not len(W):
        return ["  （頭数不足）"]
    wc = W[W["ワイド確率"] >= WIDE_PMIN].sort_values("度合い", ascending=False)
    if len(wc) < WIDE_SHOW:
        wc = pd.concat([wc, W.drop(wc.index).sort_values("ワイド確率", ascending=False)])
    bought = {b["買い目"] for b in bets if b["券種"] == "ワイド"}
    out = ["  組     度合い  確率  市場  補正後  結果"]
    for _, w in wc.head(WIDE_SHOW).iterrows():
        a_, b_ = map(int, str(w["組み合わせ"]).split("-"))
        hit = fin.get(a_, 99) <= 3 and fin.get(b_, 99) <= 3
        pay = WPAY.get((rid, str(w["組み合わせ"])))
        out.append(f"  {w['組み合わせ']:<6} {w['度合い']:>5.2f}  {w['ワイド確率']:.2f}  {w['市場のワイド確率']:.2f}  {w['補正後の確率']:.2f}   "
                   + (("○" + (f" {pay:.0f}円" if pay else "")) if hit else "×") + (" ←買い" if w["組み合わせ"] in bought else ""))
    return out
def bet_text(bets):
    if not bets:
        return "なし"
    return " / ".join(f"{b['券種']}{b['買い目']} " + (f"的中 {b['払戻']:,.0f}円" if b["的中"] and b["払戻"] == b["払戻"] else ("的中" if b["的中"] else "はずれ")) for b in bets)
for x in RACES:
    n, o, fin = x["new"].copy(), x["old"].set_index("馬番"), x["fin"]
    t = pd.DataFrame({"印": n["印"], "番": n["馬番"].astype(int), "馬名": n["馬名"].astype(str).str[:8],
                      "新 1着率": n["1着確率"].map(lambda v: f"{v:.1%}"), "旧 1着率": n["馬番"].map(o["1着確率"]).map(lambda v: f"{v:.1%}"),
                      "旧の印": n["馬番"].map(o["印"]).fillna(""), "単勝(4分前)": n["単勝"],
                      "新 期待値": n["期待値"].round(2), "旧 期待値": n["馬番"].map(o["期待値"]).round(2),
                      "着順": n["馬番"].astype(int).map(fin).map(lambda v: "" if v != v else f"{int(v)}着")})
    print(f"\n━━━━ {x['venue']} {x['R']}R（{x['n']}頭）━━━━")
    print(t.head(8).to_string(index=False))
    print(f"▼ 新しいモデルの買い目: {bet_text(x['bets_new'])}")
    print(f"▼ 旧モデルの買い目:     {bet_text(x['bets_old'])}")
    print("新しいモデルのワイド候補"); print("\n".join(wide_lines(x["Wn"], x["bets_new"], x["rid"], fin)))
    print("旧モデルのワイド候補"); print("\n".join(wide_lines(x["Wo"], x["bets_old"], x["rid"], fin)))

## 1日のまとめ

In [ ]:
def tally(key, kind=None):
    bs = [b for x in RACES for b in x[key] if kind is None or b["券種"] == kind]
    n = len(bs); hit = sum(b["的中"] for b in bs); pay = np.nansum([b["払戻"] for b in bs])
    unk = sum(1 for b in bs if b["的中"] and b["払戻"] != b["払戻"])
    return {"点数": n, "的中": hit, "投資": n * 100, "払戻": int(pay), "回収率": f"{pay / (n * 100):.0%}" if n else "-",
            "備考": f"払戻不明の的中 {unk}点" if unk else ""}
rows = []
for lab, key in [("新しいモデル", "bets_new"), ("旧モデル", "bets_old")]:
    for kind in ["単勝", "ワイド", None]:
        rows.append({"": f"{lab} {kind or '合計'}", **tally(key, kind)})
if len(ACT):
    for kind in ["単勝", "ワイド", None]:
        a = ACT if kind is None else ACT[ACT["券種"] == kind]
        rows.append({"": f"参考: 今日の通知（実際）{kind or '合計'}", "点数": len(a), "的中": int(a["的中"].fillna(False).astype(bool).sum()),
                     "投資": len(a) * 100, "払戻": int(a["払戻"].fillna(0).sum()),
                     "回収率": f"{a['払戻'].fillna(0).sum() / (len(a) * 100):.0%}" if len(a) else "-", "備考": ""})
print("【自動予想と同じ買い方での成績】4分前のオッズで判断・払戻は確定")
print(pd.DataFrame(rows).to_string(index=False))

# 買い基準を振った場合（同じ日のレースで）
def fav_rule(key, th):
    pts = pay = hit = 0
    for x in RACES:
        out = x[key]; f = out.iloc[0]
        if f["期待値"] >= th:
            pts += 1; n = int(f["馬番"])
            if x["fin"].get(n) == 1:
                hit += 1; pay += x["fodds"].get(n, 0) * 100
    return pts, hit, pay
def wide_rule(wkey, th):
    pts = pay = hit = unk = 0
    for x in RACES:
        W = x[wkey]
        if W is None or not len(W):
            continue
        c_ = W[(W["度合い"] >= th) & (W["ワイド確率"] >= WIDE_PMIN)].sort_values("度合い", ascending=False).head(1)
        for _, w in c_.iterrows():
            pts += 1
            a_, b_ = map(int, str(w["組み合わせ"]).split("-"))
            if x["fin"].get(a_, 99) <= 3 and x["fin"].get(b_, 99) <= 3:
                hit += 1; p_ = WPAY.get((x["rid"], str(w["組み合わせ"])))
                pay += p_ or 0; unk += p_ is None
    return pts, hit, pay
rows = []
for th in [1.0, 1.05, 1.1, 1.2]:
    r_ = {"単勝 ◎で期待値": f"{th:.2f}以上" + ("（本番）" if th == TAN_TH else "")}
    for lab, key in [("新", "new"), ("旧", "old")]:
        p_, h_, y_ = fav_rule(key, th)
        r_[lab] = f"{p_}点 {h_}的中 {y_ / (p_ * 100):.0%}" if p_ else "0点"
    rows.append(r_)
print("\n【単勝の買い基準を振ると】"); print(pd.DataFrame(rows).to_string(index=False))
rows = []
for th in [1.3, 1.4, 1.5, 1.6, 1.8]:
    r_ = {"ワイド 度合い": f"{th:.1f}以上" + ("（本番）" if th == WIDE_RATIO else "") + f"・確率{WIDE_PMIN}以上・1点"}
    for lab, key in [("新", "Wn"), ("旧", "Wo")]:
        p_, h_, y_ = wide_rule(key, th)
        r_[lab] = f"{p_}点 {h_}的中 {y_ / (p_ * 100):.0%}" if p_ else "0点"
    rows.append(r_)
print("\n【ワイドの買い基準を振ると】"); print(pd.DataFrame(rows).to_string(index=False))

# 印の成績と、勝ち馬につけた確率
def mark_rate(key, mark, top):
    k = [x for x in RACES]
    hit = sum(any((x["fin"].get(int(n), 99) <= top) for n in x[key].loc[x[key]["印"] == mark, "馬番"]) for x in k)
    return f"{hit}/{len(k)}（{hit / max(len(k), 1):.0%}）"
print("\n【印の成績】")
print(pd.DataFrame([{"": lab, "◎の1着": mark_rate(key, "◎", 1), "◎の3着内": mark_rate(key, "◎", 3), "○の3着内": mark_rate(key, "○", 3)}
                    for lab, key in [("新しいモデル", "new"), ("旧モデル", "old")]]).to_string(index=False))
print(f"◎が新旧で違ったレース: {sum(int(x['new'].iloc[0]['馬番']) != int(x['old'].iloc[0]['馬番']) for x in RACES)}/{len(RACES)}")
def wp(x, key):
    w1 = x["fin"][x["fin"] == 1].index
    if not len(w1): return np.nan
    s = x[key].set_index("馬番")["1着確率"]
    return float(s.get(int(w1[0]), np.nan))
def wm(x):
    w1 = x["fin"][x["fin"] == 1].index
    inv = 1 / x["rec"].set_index("馬番")["単勝_予想時"].astype(float)
    return float((inv / inv.sum()).get(int(w1[0]), np.nan)) if len(w1) else np.nan
lm = -np.log(np.clip([wm(x) for x in RACES], 1e-6, None)).mean()
print("\n【勝ち馬につけた確率】（対数損失。小さいほど良い。1日分なので偶然で大きく動く）")
print(f"  市場（4分前のオッズ）: {lm:.4f}")
for lab, key in [("旧モデル", "old"), ("新しいモデル", "new")]:
    v = -np.log(np.clip([wp(x, key) for x in RACES], 1e-6, None)).mean()
    print(f"  {lab}: {v:.4f}（市場との差 {lm - v:+.4f}）")

# 保存（あとで見返す用）
H = []
for x in RACES:
    o = x["old"].set_index("馬番")
    for _, r in x["new"].iterrows():
        H.append({"race_id": x["rid"], "場": x["venue"], "R": x["R"], "馬番": int(r["馬番"]), "馬名": r["馬名"], "新の印": r["印"], "旧の印": o["印"].get(r["馬番"], ""),
                  "単勝_4分前": r["単勝"], "新_1着確率": r["1着確率"], "旧_1着確率": o["1着確率"].get(r["馬番"]), "新_期待値": r["期待値"], "旧_期待値": o["期待値"].get(r["馬番"]),
                  "着順": x["fin"].get(int(r["馬番"])), "単勝_確定": x["fodds"].get(int(r["馬番"]))})
outdir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
pd.DataFrame(H).to_csv(f"{outdir}/replay_{DATE}.csv", index=False, encoding="utf-8-sig")
print(f"\n保存: replay_{DATE}.csv（馬ごとの新旧の予想と結果）")
print("※ 1日分は数十レースなので、回収率は偶然で大きく動く。良し悪しの判断は記録チェックで2〜4週間分を見て行う")